# P0-corrected ACLD experimental campaign

**Purpose.** This copy preserves the original notebook and applies methodological P0 corrections before any new manuscript claims are generated.

P0 changes in this copy:
1. Edge-F1 ground truth is always the exact Tarjan bridge set of the realized next-step viable graph; the sampled approximation is no longer used for manuscript metrics.
2. Scenario-comparison runtime is explicitly labelled as post-shared-preprocessing / incremental latency. ACLD end-to-end pipeline latency is recorded separately from `run_acld` and remains independently benchmarked by C2.
3. Statistical effect-size output uses `paired_sign_effect` rather than the misleading `cliffs_delta_paired` label. D1 is changed to a scenario-balanced paired analysis with Holm correction.
4. Corrected outputs are written to `acld_full_experiment_outputs_P0_corrected` so legacy and corrected rows cannot be mixed by resume logic.
5. A GCS ON/OFF sensitivity block is added to test whether the method depends on a ground-station node.
6. A payload-only communication-overhead estimator is added as an explicitly assumption-based lower-bound calculation, not as measured network traffic.

> IMPORTANT: All manuscript Edge-F1 values and comparative runtime claims must be regenerated from this corrected campaign before reuse.

- P0.1: aggregate tables preserve all runtime scopes (`runtime_incremental_*`, `acld_pipeline_*`, `shared_context_*`) instead of silently collapsing to the legacy alias.

- P0.2: method-level runtimes are explicitly non-comparable diagnostics; full `acld_pipeline_*` is populated only for `acld_v3`, and manuscript scalability relies on C2 full-core timing.

- P0.3: exact localization retains all valid predicted pairs (a predicted pair absent from the realized next graph is FP), no-support steps are undefined rather than F1=0, and the primary episode Edge-F1 is micro-aggregated from exact TP/FP/FN before seed-level statistics.

9. P0.4 adds an environment/provenance manifest so CPU/OS/Python/package/source claims are captured from the actual rerun environment rather than manuscript prose.


# ACLD-v3 Full Experimental Campaign Notebook

Bu notebook, makaledeki **bütün majör sorunları deneyle kapatmak** için tasarlandı.  
Her ana deney **ayrı hücrede** çalışır ve yalnız deneyle desteklenen iddialar bırakır.

## Deney paketleri
- **A1** Ana detection/localization campaign (Nakagami, NoJam / 5 dB / 15 dB)
- **A2** Kanal bozulumu ve Rician-stress campaign
- **A3** ACLD ablation campaign
- **B1** Closed-loop mitigation campaign (nominal)
- **B2** Closed-loop mitigation campaign (harsh stress)
- **C1** Accuracy scaling campaign
- **C2** Runtime scaling campaign
- **D1** İstatistiksel anlamlılık analizi
- **D2** Prevalence-controlled jamming analizi
- **D3** Calibration / reliability analizi
- **D4** Failure-mode analizi
- **E** Makale tabloları ve figürleri

## Çalıştırma sırası
Notebook baştan sona sıralı çalıştırılmalıdır.  
Tam kampanya uzundur; önce küçük bir pilot deneme yapmak isterseniz aşağıdaki `SEEDS_*` ve `HORIZON_*` ayarlarını düşürebilirsiniz.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Set, Iterable
import math
import time

import os
import csv
import json
import copy
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
from scipy.special import gammaincc
from scipy.spatial.distance import cdist
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

# ============================================================
# Configuration
# ============================================================

@dataclass
class ACLDConfig:
    # --- timing ---
    dt_ctrl: float = 0.05
    dt_sim: float = 0.05

    # --- swarm ---
    n_uavs: int = 150
    include_ground_station: bool = True
    ground_station_pos: np.ndarray = field(default_factory=lambda: np.array([0.0, 0.0, 0.0], dtype=float))

    # --- physical limits ---
    v_max: float = 23.0
    a_max: float = 60.0
    j_max: float = 100.0
    d_safe: float = 2.0  # collision-avoidance minimum separation (example)

    # --- channel (default UAV-UAV) ---
    m_fading: float = 2.5
    snr_0_db: float = 22.0
    d_0: float = 100.0
    path_loss_exp: float = 2.4
    gamma_0_db: float = 10.0
    p_min: float = 0.90

    # --- UAV-GS channel profile (paper says Free-space LoS for UAV-GS) ---
    gs_m_fading: float = 3.5
    gs_snr_0_db: float = 25.0
    gs_path_loss_exp: float = 2.0

    # --- adaptive gains ---
    alpha: float = 0.0
    beta: float = 0.10
    gamma: float = 0.20
    delta: float = 0.10
    zeta: float = 0.05

    # --- base range ---
    cthr_base: float = 100.0

    # --- numerical guards ---
    min_cthr: float = 5.0
    max_cthr_multiplier: float = 2.0
    lambda2_partition_threshold: float = 1e-3
    risk_band_fraction: float = 0.97

    # --- energy (optional simple simulation bookkeeping) ---
    e_max: float = 1000.0
    e_min_frac: float = 0.15

    # --- jamming / SINR params ---
    # Controlled stress mode: set jnr_db = None (nominal), or e.g. 5 / 15 dB
    jnr_db: Optional[float] = None
    # For detailed jammer model (optional):
    jammer_tx_dbm: Optional[float] = None
    jammer_pos: Optional[np.ndarray] = None
    jammer_gain_db: float = 0.0
    rx_gain_db: float = 0.0
    noise_figure_db: float = 6.0
    bandwidth_hz: float = 20e6

    # --- simulation area ---
    density_scale_mode: str = "medium"  # "sparse" | "medium" | "dense"
    seed: int = 42

    @property
    def snr_0_lin(self) -> float:
        return 10 ** (self.snr_0_db / 10.0)

    @property
    def gamma_0_lin(self) -> float:
        return 10 ** (self.gamma_0_db / 10.0)

    @property
    def gs_snr_0_lin(self) -> float:
        return 10 ** (self.gs_snr_0_db / 10.0)

    @property
    def n_total(self) -> int:
        return self.n_uavs + (1 if self.include_ground_station else 0)

    @property
    def gs_index(self) -> Optional[int]:
        return self.n_uavs if self.include_ground_station else None

    def density_half_side(self) -> float:
        mode = self.density_scale_mode.lower()
        if mode == "sparse":
            scale = 1.5
        elif mode == "dense":
            scale = 0.3
        else:
            scale = 0.8  # medium
        return scale * self.cthr_base

# ============================================================
# State
# ============================================================

@dataclass
class SwarmState:
    pos: np.ndarray          # (n,3)
    vel: np.ndarray          # (n,3)
    acc_cmd: np.ndarray      # (n,3)
    jerk: np.ndarray         # (n,3)
    energy: np.ndarray       # (n,)
    channel_headroom_db: np.ndarray  # (n,)

    @staticmethod
    def random_init(cfg: ACLDConfig, rng: np.random.Generator) -> "SwarmState":
        n = cfg.n_uavs
        half_side = cfg.density_half_side()

        pos = rng.uniform(-half_side, half_side, size=(n, 3))
        vel = rng.normal(0.0, cfg.v_max / 4.0, size=(n, 3))
        vel = _clip_rows_norm(vel, cfg.v_max)

        acc_cmd = rng.normal(0.0, cfg.a_max / 6.0, size=(n, 3))
        acc_cmd = _clip_rows_norm(acc_cmd, cfg.a_max)

        jerk = rng.normal(0.0, cfg.j_max / 6.0, size=(n, 3))
        jerk = _clip_rows_norm(jerk, cfg.j_max)

        energy = np.full(n, cfg.e_max, dtype=float)
        channel_headroom_db = np.full(n, 0.0, dtype=float)

        return SwarmState(
            pos=pos.astype(float),
            vel=vel.astype(float),
            acc_cmd=acc_cmd.astype(float),
            jerk=jerk.astype(float),
            energy=energy.astype(float),
            channel_headroom_db=channel_headroom_db.astype(float),
        )

    def step_random_kinematics(self, cfg: ACLDConfig, rng: np.random.Generator) -> None:
        """
        Simple bounded kinematics update for Monte-Carlo simulations.
        This is a placeholder simulation model, not a flight controller.
        """
        dt = cfg.dt_sim

        # Random jerk perturbation
        jerk_noise = rng.normal(0.0, cfg.j_max / 20.0, size=self.jerk.shape)
        self.jerk = _clip_rows_norm(self.jerk + jerk_noise, cfg.j_max)

        # Update control acceleration
        self.acc_cmd = _clip_rows_norm(self.acc_cmd + self.jerk * dt, cfg.a_max)

        # Update velocity
        self.vel = _clip_rows_norm(self.vel + self.acc_cmd * dt, cfg.v_max)

        # Update position
        self.pos = self.pos + self.vel * dt

        # Simple energy bookkeeping
        # (Not physically exact; enough for state completeness)
        speed = np.linalg.norm(self.vel, axis=1)
        acc = np.linalg.norm(self.acc_cmd, axis=1)
        power_proxy = 5.0 + 0.2 * speed + 0.02 * acc**2  # arbitrary proxy
        self.energy = np.maximum(0.0, self.energy - power_proxy * dt)

    def copy(self) -> "SwarmState":
        return SwarmState(
            pos=self.pos.copy(),
            vel=self.vel.copy(),
            acc_cmd=self.acc_cmd.copy(),
            jerk=self.jerk.copy(),
            energy=self.energy.copy(),
            channel_headroom_db=self.channel_headroom_db.copy(),
        )

# ============================================================
# Result containers
# ============================================================

@dataclass
class ACLDResult:
    A_pred_guaranteed: np.ndarray
    A_pred_viable: np.ndarray
    d_pred_uav: np.ndarray
    d_cur_uav: np.ndarray
    theta_minus_uav: np.ndarray
    theta_plus_uav: np.ndarray
    theta_minus_gs: Optional[np.ndarray]
    theta_plus_gs: Optional[np.ndarray]
    s_pred_uav: np.ndarray
    s_cur_uav: np.ndarray
    components: List[Set[int]]
    comp_of: Dict[int, int]
    risky_inter: List[Tuple[int, int]]
    risky_intra: List[Tuple[int, int]]
    risky_s_uav: List[Tuple[int, int]]
    risky_uav_uav: List[Tuple[int, int]]
    predicted_viable_bridges: List[Tuple[int, int]]
    selected_critical_edges: List[Tuple[int, int]]
    n_components_pred_viable: int
    lambda2_current: float
    lambda2_pred_viable: float
    W_current: np.ndarray
    L_current: np.ndarray

# ============================================================
# Core ACLD
# ============================================================

class ACLDCore:
    def __init__(self, cfg: ACLDConfig):
        self.cfg = cfg

    # --------------------------
    # Geometry utilities
    # --------------------------
    def _pairwise_uav_dist(self, pos: np.ndarray) -> np.ndarray:
        return cdist(pos, pos, metric="euclidean")

    def _uav_gs_dist(self, pos: np.ndarray) -> np.ndarray:
        gs = self.cfg.ground_station_pos.reshape(1, 3)
        return np.linalg.norm(pos - gs, axis=1)

    def _dispersion_stats(self, d_uav: np.ndarray) -> Tuple[float, float]:
        n = d_uav.shape[0]
        iu = np.triu_indices(n, k=1)
        vals = d_uav[iu]
        if vals.size == 0:
            return 0.0, 1.0
        bar_d = float(np.mean(vals))
        sigma_d = float(np.std(vals))
        return sigma_d, max(bar_d, 1e-9)

    def _heading_diff_matrix(self, vel: np.ndarray) -> np.ndarray:
        """
        Vectorized absolute angle difference in [0, pi] using 3D vectors.
        If a vector norm is ~0, angle term defaults to 0 for that row/column pair.
        """
        norms = np.linalg.norm(vel, axis=1)
        safe = np.where(norms > 1e-9, norms, 1.0)
        unit = vel / safe[:, None]
        cosang = np.clip(unit @ unit.T, -1.0, 1.0)
        H = np.arccos(cosang)
        invalid = norms <= 1e-9
        if np.any(invalid):
            H[invalid, :] = 0.0
            H[:, invalid] = 0.0
        np.fill_diagonal(H, 0.0)
        return H

    # --------------------------
    # Adaptive thresholds
    # --------------------------
    def adaptive_threshold_uav_uav(
        self,
        state: SwarmState,
        d_uav: np.ndarray,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Returns cthr_ij, theta_minus_ij, theta_plus_ij
        """
        cfg = self.cfg
        n = cfg.n_uavs

        sigma_d, bar_d = self._dispersion_stats(d_uav)
        rel_speed = cdist(state.vel, state.vel, metric="euclidean")
        rel_acc = cdist(state.acc_cmd, state.acc_cmd, metric="euclidean")
        rel_jerk = cdist(state.jerk, state.jerk, metric="euclidean")
        dtheta = self._heading_diff_matrix(state.vel)

        term = (
            1.0
            + cfg.alpha * (sigma_d / bar_d)
            - cfg.beta * (rel_speed / (2.0 * cfg.v_max + 1e-12))
            - cfg.gamma * (dtheta / math.pi)
            - cfg.delta * (rel_acc / (2.0 * cfg.a_max + 1e-12))
            - cfg.zeta * (rel_jerk / (2.0 * cfg.j_max + 1e-12))
        )
        cthr = cfg.cthr_base * term

        cthr = np.clip(cthr, cfg.min_cthr, cfg.max_cthr_multiplier * cfg.cthr_base)
        np.fill_diagonal(cthr, 0.0)

        theta_plus = cthr
        theta_minus = cfg.risk_band_fraction * cthr
        np.fill_diagonal(theta_minus, 0.0)
        np.fill_diagonal(theta_plus, 0.0)

        return cthr, theta_minus, theta_plus

    def adaptive_threshold_uav_gs(self, state: SwarmState, d_uav: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Eq. cthradp_gs in paper (no heading term).
        Returns cthr_is, theta_minus_is, theta_plus_is of shape (n_uavs,)
        """
        cfg = self.cfg
        sigma_d, bar_d = self._dispersion_stats(d_uav)

        speed = np.linalg.norm(state.vel, axis=1)
        acc = np.linalg.norm(state.acc_cmd, axis=1)
        jerk = np.linalg.norm(state.jerk, axis=1)

        term = (
            1.0
            + cfg.alpha * (sigma_d / bar_d)
            - cfg.beta * (speed / (cfg.v_max + 1e-12))
            - cfg.delta * (acc / (cfg.a_max + 1e-12))
            - cfg.zeta * (jerk / (cfg.j_max + 1e-12))
        )
        cthr = cfg.cthr_base * term
        cthr = np.clip(cthr, cfg.min_cthr, cfg.max_cthr_multiplier * cfg.cthr_base)
        theta_plus = cthr
        theta_minus = cfg.risk_band_fraction * cthr
        return cthr, theta_minus, theta_plus

    # --------------------------
    # Channel model (Nakagami-m)
    # --------------------------
    def _nb_dbm(self) -> float:
        cfg = self.cfg
        return -174.0 + 10.0 * math.log10(cfg.bandwidth_hz) + cfg.noise_figure_db

    def _detailed_jnr_db_per_receiver(self, pos: np.ndarray) -> Optional[np.ndarray]:
        cfg = self.cfg
        if cfg.jammer_tx_dbm is None or cfg.jammer_pos is None:
            return None

        nb_dbm = self._nb_dbm()
        d = np.linalg.norm(pos - cfg.jammer_pos.reshape(1, 3), axis=1)
        d = np.maximum(d, 1.0)

        # Simple log-distance path loss (consistent enough for stress simulation)
        pl_db = 20.0 * np.log10(4 * np.pi * cfg.d_0 / 0.0517) + 10.0 * cfg.path_loss_exp * np.log10(d / cfg.d_0)
        p_rx_j_dbm = cfg.jammer_tx_dbm + cfg.jammer_gain_db + cfg.rx_gain_db - pl_db
        jnr_db = p_rx_j_dbm - nb_dbm
        return jnr_db

    def _receiver_jnr_lin(self, pos: np.ndarray) -> Optional[np.ndarray]:
        cfg = self.cfg

        if cfg.jnr_db is not None:
            # controlled receiver-referenced stress (paper-friendly)
            return np.full(cfg.n_uavs, 10 ** (cfg.jnr_db / 10.0), dtype=float)

        jnr_db_per_rx = self._detailed_jnr_db_per_receiver(pos)
        if jnr_db_per_rx is None:
            return None
        return 10 ** (jnr_db_per_rx / 10.0)

    def _nakagami_success_from_mean_snr(self, mean_snr: np.ndarray, m: float) -> np.ndarray:
        cfg = self.cfg
        x = m * (cfg.gamma_0_lin / np.maximum(mean_snr, 1e-12))
        # Γ(m,x)/Γ(m) = gammaincc(m,x)
        s = gammaincc(m, x)
        return np.clip(s, 0.0, 1.0)

    def success_probability_uav_uav(self, d_uav: np.ndarray, pos_for_jammer: Optional[np.ndarray] = None) -> np.ndarray:
        cfg = self.cfg
        d = np.maximum(d_uav, 1e-9)

        mean_snr = cfg.snr_0_lin * (cfg.d_0 / d) ** cfg.path_loss_exp

        # Jamming-aware SINR if configured
        if pos_for_jammer is None:
            pos_for_jammer = None

        jnr_lin_per_rx = None
        if pos_for_jammer is not None:
            jnr_lin_per_rx = self._receiver_jnr_lin(pos_for_jammer)

        if jnr_lin_per_rx is not None:
            # conservative link-level j_lin_ij = max(j_i, j_j)
            ji = jnr_lin_per_rx.reshape(-1, 1)
            jj = jnr_lin_per_rx.reshape(1, -1)
            j_lin_ij = np.maximum(ji, jj)
            mean_snr = mean_snr / (1.0 + j_lin_ij)

        s = self._nakagami_success_from_mean_snr(mean_snr, cfg.m_fading)
        np.fill_diagonal(s, 0.0)
        return s

    def success_probability_uav_gs(self, d_gs: np.ndarray, pos: np.ndarray) -> np.ndarray:
        cfg = self.cfg
        d = np.maximum(d_gs, 1e-9)

        mean_snr = cfg.gs_snr_0_lin * (cfg.d_0 / d) ** cfg.gs_path_loss_exp

        jnr_lin_per_rx = self._receiver_jnr_lin(pos)
        if jnr_lin_per_rx is not None:
            # receiver is UAV i for UAV-GS link
            mean_snr = mean_snr / (1.0 + jnr_lin_per_rx)

        s = self._nakagami_success_from_mean_snr(mean_snr, cfg.gs_m_fading)
        return np.clip(s, 0.0, 1.0)

    # --------------------------
    # Predicted adjacency / risk sets
    # --------------------------
    def _predict_positions_one_step(self, state: SwarmState) -> np.ndarray:
        return state.pos + state.vel * self.cfg.dt_ctrl

    def _build_pred_matrices(
        self, state: SwarmState
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray]]:
        """
        Returns:
          d_cur_uav, d_pred_uav,
          theta_minus_uav, theta_plus_uav,
          s_cur_uav, s_pred_uav,
          d_cur_gs, d_pred_gs, theta_minus_gs, theta_plus_gs
        """
        d_cur_uav = self._pairwise_uav_dist(state.pos)
        pred_pos = self._predict_positions_one_step(state)
        d_pred_uav = self._pairwise_uav_dist(pred_pos)

        _, theta_minus_uav, theta_plus_uav = self.adaptive_threshold_uav_uav(state, d_cur_uav)

        s_cur_uav = self.success_probability_uav_uav(d_cur_uav, pos_for_jammer=state.pos)
        s_pred_uav = self.success_probability_uav_uav(d_pred_uav, pos_for_jammer=pred_pos)

        if self.cfg.include_ground_station:
            d_cur_gs = self._uav_gs_dist(state.pos)
            d_pred_gs = self._uav_gs_dist(pred_pos)
            _, theta_minus_gs, theta_plus_gs = self.adaptive_threshold_uav_gs(state, d_cur_uav)
        else:
            d_cur_gs = d_pred_gs = theta_minus_gs = theta_plus_gs = None

        return (
            d_cur_uav, d_pred_uav,
            theta_minus_uav, theta_plus_uav,
            s_cur_uav, s_pred_uav,
            d_cur_gs, d_pred_gs, theta_minus_gs, theta_plus_gs
        )

    def _build_pred_adjacency(
        self,
        d_pred_uav: np.ndarray,
        theta_minus_uav: np.ndarray,
        theta_plus_uav: np.ndarray,
        s_pred_uav: np.ndarray,
        d_pred_gs: Optional[np.ndarray],
        theta_minus_gs: Optional[np.ndarray],
        theta_plus_gs: Optional[np.ndarray],
        s_pred_gs: Optional[np.ndarray],
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Build predicted guaranteed and viable adjacency on V' (including GS if enabled).
        """
        cfg = self.cfg
        n_uav = cfg.n_uavs
        n_total = cfg.n_total

        A_minus = np.zeros((n_total, n_total), dtype=np.uint8)
        A_plus = np.zeros((n_total, n_total), dtype=np.uint8)

        # UAV-UAV edges
        viable_uav = (d_pred_uav <= theta_plus_uav) & (s_pred_uav >= cfg.p_min)
        guar_uav = (d_pred_uav <= theta_minus_uav) & (s_pred_uav >= cfg.p_min)

        np.fill_diagonal(viable_uav, False)
        np.fill_diagonal(guar_uav, False)

        A_plus[:n_uav, :n_uav] = viable_uav.astype(np.uint8)
        A_minus[:n_uav, :n_uav] = guar_uav.astype(np.uint8)

        # UAV-GS edges
        if cfg.include_ground_station:
            assert d_pred_gs is not None and theta_plus_gs is not None and theta_minus_gs is not None and s_pred_gs is not None
            sidx = cfg.gs_index
            viable_gs = (d_pred_gs <= theta_plus_gs) & (s_pred_gs >= cfg.p_min)
            guar_gs = (d_pred_gs <= theta_minus_gs) & (s_pred_gs >= cfg.p_min)

            for i in range(n_uav):
                if viable_gs[i]:
                    A_plus[i, sidx] = 1
                    A_plus[sidx, i] = 1
                if guar_gs[i]:
                    A_minus[i, sidx] = 1
                    A_minus[sidx, i] = 1

        return A_minus, A_plus

    # --------------------------
    # Pseudocode-equivalent graph routines
    # --------------------------
    @staticmethod
    def _adj_list_from_adjacency(A: np.ndarray) -> List[List[int]]:
        n = A.shape[0]
        adj = []
        for i in range(n):
            adj.append(np.flatnonzero(A[i]).tolist())
        return adj

    @staticmethod
    def kneighbors(adj: List[List[int]], seed: int, k_max: int) -> Set[int]:
        seen: Set[int] = {seed}
        q: List[Tuple[int, int]] = [(seed, 0)]
        head = 0
        while head < len(q):
            u, depth = q[head]
            head += 1
            if depth == k_max:
                continue
            for v in adj[u]:
                if v not in seen:
                    seen.add(v)
                    q.append((v, depth + 1))
        return seen

    @classmethod
    def predict_components(cls, A_guaranteed: np.ndarray, k_max: Optional[int] = None) -> Tuple[List[Set[int]], Dict[int, int]]:
        n = A_guaranteed.shape[0]
        if k_max is None:
            k_max = n  # full BFS as in manuscript note

        adj = cls._adj_list_from_adjacency(A_guaranteed)

        unseen = set(range(n))
        comps: List[Set[int]] = []
        comp_of: Dict[int, int] = {}

        while unseen:
            i = next(iter(unseen))
            C = cls.kneighbors(adj, i, k_max=k_max)
            idx = len(comps)
            for u in C:
                comp_of[u] = idx
            comps.append(C)
            unseen -= C

        return comps, comp_of

    @staticmethod
    def catalogue_risky_links(
        A_pred_viable: np.ndarray,
        comp_of: Dict[int, int],
        cfg: ACLDConfig,
        d_pred_uav: np.ndarray,
        theta_minus_uav: np.ndarray,
        theta_plus_uav: np.ndarray,
        d_pred_gs: Optional[np.ndarray],
        theta_minus_gs: Optional[np.ndarray],
        theta_plus_gs: Optional[np.ndarray],
    ) -> Tuple[List[Tuple[int, int]], List[Tuple[int, int]]]:
        """
        Risk-band = viable but not guaranteed (theta_minus < d <= theta_plus, and viable already ensured).
        Returns (inter, intra)
        """
        n_total = A_pred_viable.shape[0]
        n_uav = cfg.n_uavs
        sidx = cfg.gs_index

        inter: List[Tuple[int, int]] = []
        intra: List[Tuple[int, int]] = []

        for i in range(n_total):
            for j in range(i + 1, n_total):
                if A_pred_viable[i, j] == 0:
                    continue

                in_risk_band = False

                if i < n_uav and j < n_uav:
                    dij = d_pred_uav[i, j]
                    if theta_minus_uav[i, j] < dij <= theta_plus_uav[i, j]:
                        in_risk_band = True
                else:
                    # UAV-GS case
                    if sidx is None:
                        continue
                    if d_pred_gs is None or theta_minus_gs is None or theta_plus_gs is None:
                        continue
                    u = i if j == sidx else j if i == sidx else None
                    if u is not None and 0 <= u < n_uav:
                        du = d_pred_gs[u]
                        if theta_minus_gs[u] < du <= theta_plus_gs[u]:
                            in_risk_band = True

                if not in_risk_band:
                    continue

                if comp_of.get(i, -1) != comp_of.get(j, -1):
                    inter.append((i, j))
                else:
                    intra.append((i, j))

        return inter, intra

    @staticmethod
    def separate_ground_links(
        risky_inter: List[Tuple[int, int]],
        gs_index: Optional[int]
    ) -> Tuple[List[Tuple[int, int]], List[Tuple[int, int]]]:
        if gs_index is None:
            return [], risky_inter.copy()

        s_uav: List[Tuple[int, int]] = []
        uav_uav: List[Tuple[int, int]] = []

        for i, j in risky_inter:
            if i == gs_index or j == gs_index:
                s_uav.append((i, j))
            else:
                uav_uav.append((i, j))
        return s_uav, uav_uav

    # --------------------------
    # Weighted graph / Laplacian / lambda2
    # --------------------------
    def _build_current_weighted_graph(self, state: SwarmState) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Build W and L on current channel-viable graph G_viable(t), including GS if enabled.
        Uses geometric candidates + viability guard + directed row-stochastic inverse-square weights + symmetrization.
        """
        cfg = self.cfg
        n_uav = cfg.n_uavs
        n_total = cfg.n_total
        sidx = cfg.gs_index

        d_cur_uav = self._pairwise_uav_dist(state.pos)
        _, theta_minus_uav, theta_plus_uav = self.adaptive_threshold_uav_uav(state, d_cur_uav)
        s_cur_uav = self.success_probability_uav_uav(d_cur_uav, pos_for_jammer=state.pos)

        d_cur_gs = theta_minus_gs = theta_plus_gs = s_cur_gs = None
        if cfg.include_ground_station:
            d_cur_gs = self._uav_gs_dist(state.pos)
            _, theta_minus_gs, theta_plus_gs = self.adaptive_threshold_uav_gs(state, d_cur_uav)
            s_cur_gs = self.success_probability_uav_gs(d_cur_gs, state.pos)

        W_tilde = np.zeros((n_total, n_total), dtype=float)

        # UAV rows
        for i in range(n_uav):
            neighbors: List[Tuple[int, float]] = []

            # UAV-UAV viable neighbors
            for j in range(n_uav):
                if i == j:
                    continue
                if d_cur_uav[i, j] <= theta_plus_uav[i, j] and s_cur_uav[i, j] >= cfg.p_min:
                    neighbors.append((j, max(d_cur_uav[i, j], cfg.d_safe)))

            # UAV-GS viable neighbor
            if cfg.include_ground_station and d_cur_gs is not None and theta_plus_gs is not None and s_cur_gs is not None:
                if d_cur_gs[i] <= theta_plus_gs[i] and s_cur_gs[i] >= cfg.p_min:
                    neighbors.append((sidx, max(d_cur_gs[i], cfg.d_safe)))

            if len(neighbors) == 0:
                continue

            invsq = np.array([1.0 / (d * d) for _, d in neighbors], dtype=float)
            denom = float(np.sum(invsq))
            if denom <= 0:
                continue

            for (k, _d), w in zip(neighbors, invsq / denom):
                W_tilde[i, k] = float(w)

        # GS row (if present)
        if cfg.include_ground_station and sidx is not None and d_cur_gs is not None and theta_plus_gs is not None and s_cur_gs is not None:
            neighbors_gs: List[Tuple[int, float]] = []
            for i in range(n_uav):
                if d_cur_gs[i] <= theta_plus_gs[i] and s_cur_gs[i] >= cfg.p_min:
                    neighbors_gs.append((i, max(d_cur_gs[i], cfg.d_safe)))

            if neighbors_gs:
                invsq = np.array([1.0 / (d * d) for _, d in neighbors_gs], dtype=float)
                denom = float(np.sum(invsq))
                if denom > 0:
                    for (i, _d), w in zip(neighbors_gs, invsq / denom):
                        W_tilde[sidx, i] = float(w)

        # Symmetrize
        W = 0.5 * (W_tilde + W_tilde.T)
        np.fill_diagonal(W, 0.0)

        D = np.diag(np.sum(W, axis=1))
        L = D - W
        return W, D, L

    @staticmethod
    def algebraic_connectivity_lambda2(L: np.ndarray) -> float:
        n = L.shape[0]
        if n <= 1:
            return 0.0
        # Dense eigvalsh is acceptable up to n~1000 in this context
        evals = np.linalg.eigvalsh(L)
        evals = np.sort(np.real(evals))
        if len(evals) < 2:
            return 0.0
        return float(max(0.0, evals[1]))

    # --------------------------
    # Baselines
    # --------------------------
    @staticmethod
    def tarjan_bridges(A: np.ndarray) -> List[Tuple[int, int]]:
        n = A.shape[0]
        adj = ACLDCore._adj_list_from_adjacency(A)

        disc = [-1] * n
        low = [-1] * n
        parent = [-1] * n
        time_counter = 0
        bridges: List[Tuple[int, int]] = []

        def dfs(u: int) -> None:
            nonlocal time_counter
            disc[u] = low[u] = time_counter
            time_counter += 1

            for v in adj[u]:
                if disc[v] == -1:
                    parent[v] = u
                    dfs(v)
                    low[u] = min(low[u], low[v])

                    if low[v] > disc[u]:
                        bridges.append((u, v) if u < v else (v, u))
                elif v != parent[u]:
                    low[u] = min(low[u], disc[v])

        for i in range(n):
            if disc[i] == -1:
                dfs(i)

        bridges = sorted(set(bridges))
        return bridges

    @staticmethod
    def edge_betweenness_brandes_unweighted(A: np.ndarray) -> Dict[Tuple[int, int], float]:
        """
        Exact Brandes edge betweenness for unweighted undirected graph.
        """
        n = A.shape[0]
        adj = ACLDCore._adj_list_from_adjacency(A)
        eb: Dict[Tuple[int, int], float] = {}

        for s in range(n):
            stack: List[int] = []
            pred: List[List[int]] = [[] for _ in range(n)]
            sigma = np.zeros(n, dtype=float)
            dist = -np.ones(n, dtype=int)

            sigma[s] = 1.0
            dist[s] = 0

            # BFS
            q = [s]
            qh = 0
            while qh < len(q):
                v = q[qh]
                qh += 1
                stack.append(v)
                for w in adj[v]:
                    if dist[w] < 0:
                        q.append(w)
                        dist[w] = dist[v] + 1
                    if dist[w] == dist[v] + 1:
                        sigma[w] += sigma[v]
                        pred[w].append(v)

            delta = np.zeros(n, dtype=float)
            while stack:
                w = stack.pop()
                for v in pred[w]:
                    if sigma[w] > 0:
                        c = (sigma[v] / sigma[w]) * (1.0 + delta[w])
                    else:
                        c = 0.0
                    e = (v, w) if v < w else (w, v)
                    eb[e] = eb.get(e, 0.0) + c
                    delta[v] += c

        # undirected graph correction
        for e in list(eb.keys()):
            eb[e] *= 0.5
        return eb

    def baseline_tarjan_snapshot(self, state: SwarmState, with_guard: bool = True) -> List[Tuple[int, int]]:
        """
        TARJAN@t: snapshot bridges on current graph.
        If with_guard=False, geometry only (cthr+ band) without probabilistic guard.
        """
        cfg = self.cfg
        n_total = cfg.n_total
        n_uav = cfg.n_uavs
        sidx = cfg.gs_index

        d_cur_uav = self._pairwise_uav_dist(state.pos)
        _, _, theta_plus_uav = self.adaptive_threshold_uav_uav(state, d_cur_uav)
        s_cur_uav = self.success_probability_uav_uav(d_cur_uav, pos_for_jammer=state.pos)

        A = np.zeros((n_total, n_total), dtype=np.uint8)

        mask_uav = (d_cur_uav <= theta_plus_uav)
        if with_guard:
            mask_uav &= (s_cur_uav >= cfg.p_min)
        np.fill_diagonal(mask_uav, False)
        A[:n_uav, :n_uav] = mask_uav.astype(np.uint8)

        if cfg.include_ground_station:
            d_cur_gs = self._uav_gs_dist(state.pos)
            _, _, theta_plus_gs = self.adaptive_threshold_uav_gs(state, d_cur_uav)
            s_cur_gs = self.success_probability_uav_gs(d_cur_gs, state.pos)

            for i in range(n_uav):
                ok = d_cur_gs[i] <= theta_plus_gs[i]
                if with_guard:
                    ok = ok and (s_cur_gs[i] >= cfg.p_min)
                if ok:
                    A[i, sidx] = A[sidx, i] = 1

        return self.tarjan_bridges(A)

    # --------------------------
    # Main ACLD step
    # --------------------------
    def run_acld(self, state: SwarmState) -> ACLDResult:
        cfg = self.cfg

        (
            d_cur_uav, d_pred_uav,
            theta_minus_uav, theta_plus_uav,
            s_cur_uav, s_pred_uav,
            d_cur_gs, d_pred_gs, theta_minus_gs, theta_plus_gs
        ) = self._build_pred_matrices(state)

        s_pred_gs = None
        if cfg.include_ground_station and d_pred_gs is not None:
            pred_pos = self._predict_positions_one_step(state)
            s_pred_gs = self.success_probability_uav_gs(d_pred_gs, pred_pos)

        A_pred_guaranteed, A_pred_viable = self._build_pred_adjacency(
            d_pred_uav=d_pred_uav,
            theta_minus_uav=theta_minus_uav,
            theta_plus_uav=theta_plus_uav,
            s_pred_uav=s_pred_uav,
            d_pred_gs=d_pred_gs,
            theta_minus_gs=theta_minus_gs,
            theta_plus_gs=theta_plus_gs,
            s_pred_gs=s_pred_gs,
        )

        components, comp_of = self.predict_components(A_pred_guaranteed, k_max=cfg.n_total)

        risky_inter, risky_intra = self.catalogue_risky_links(
            A_pred_viable=A_pred_viable,
            comp_of=comp_of,
            cfg=cfg,
            d_pred_uav=d_pred_uav,
            theta_minus_uav=theta_minus_uav,
            theta_plus_uav=theta_plus_uav,
            d_pred_gs=d_pred_gs,
            theta_minus_gs=theta_minus_gs,
            theta_plus_gs=theta_plus_gs,
        )

        risky_s_uav, risky_uav_uav = self.separate_ground_links(risky_inter, cfg.gs_index)

        W_current, D_current, L_current = self._build_current_weighted_graph(state)
        lambda2_current = self.algebraic_connectivity_lambda2(L_current)

        # predicted viable Laplacian (binary) for diagnostic only
        D_pred = np.diag(np.sum(A_pred_viable.astype(float), axis=1))
        L_pred_viable = D_pred - A_pred_viable.astype(float)
        lambda2_pred_viable = self.algebraic_connectivity_lambda2(L_pred_viable)
        n_components_pred_viable, _ = connectivity_components_from_adjacency(A_pred_viable)
        predicted_viable_bridges = self.tarjan_bridges(A_pred_viable)
        selected_critical_edges = sorted(set(predicted_viable_bridges))

        return ACLDResult(
            A_pred_guaranteed=A_pred_guaranteed,
            A_pred_viable=A_pred_viable,
            d_pred_uav=d_pred_uav,
            d_cur_uav=d_cur_uav,
            theta_minus_uav=theta_minus_uav,
            theta_plus_uav=theta_plus_uav,
            theta_minus_gs=theta_minus_gs,
            theta_plus_gs=theta_plus_gs,
            s_pred_uav=s_pred_uav,
            s_cur_uav=s_cur_uav,
            components=components,
            comp_of=comp_of,
            risky_inter=risky_inter,
            risky_intra=risky_intra,
            risky_s_uav=risky_s_uav,
            risky_uav_uav=risky_uav_uav,
            predicted_viable_bridges=predicted_viable_bridges,
            selected_critical_edges=selected_critical_edges,
            n_components_pred_viable=n_components_pred_viable,
            lambda2_current=lambda2_current,
            lambda2_pred_viable=lambda2_pred_viable,
            W_current=W_current,
            L_current=L_current,
        )

# ============================================================
# Simulation / Evaluation helpers
# ============================================================

def _clip_rows_norm(X: np.ndarray, max_norm: float) -> np.ndarray:
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    scale = np.ones_like(norms)
    mask = norms > max_norm
    scale[mask] = max_norm / np.maximum(norms[mask], 1e-12)
    return X * scale


def make_rng(seed: int) -> np.random.Generator:
    return np.random.default_rng(seed)


@dataclass
class StepLog:
    t: float
    lambda2_current: float
    partition_now: bool
    n_components_pred_guar: int
    n_risky_inter: int
    n_risky_intra: int
    n_risky_s_uav: int
    n_risky_uav_uav: int
    runtime_ms: float


def run_episode(
    cfg: ACLDConfig,
    horizon_s: float = 10.0,
    seed: Optional[int] = None,
) -> Tuple[List[StepLog], List[ACLDResult]]:
    rng = make_rng(cfg.seed if seed is None else seed)
    state = SwarmState.random_init(cfg, rng)
    acld = ACLDCore(cfg)

    T = int(round(horizon_s / cfg.dt_sim))
    logs: List[StepLog] = []
    results: List[ACLDResult] = []

    t_now = 0.0
    for _ in range(T):
        t0 = time.perf_counter()
        res = acld.run_acld(state)
        t1 = time.perf_counter()
        runtime_ms = (t1 - t0) * 1000.0

        partition_now = res.lambda2_current < cfg.lambda2_partition_threshold
        logs.append(
            StepLog(
                t=t_now,
                lambda2_current=res.lambda2_current,
                partition_now=partition_now,
                n_components_pred_guar=len(res.components),
                n_risky_inter=len(res.risky_inter),
                n_risky_intra=len(res.risky_intra),
                n_risky_s_uav=len(res.risky_s_uav),
                n_risky_uav_uav=len(res.risky_uav_uav),
                runtime_ms=runtime_ms,
            )
        )
        results.append(res)

        # Advance simulation
        state.step_random_kinematics(cfg, rng)
        t_now += cfg.dt_sim

    return logs, results


def summarize_episode(logs: List[StepLog]) -> Dict[str, float]:
    lambda2_vals = np.array([x.lambda2_current for x in logs], dtype=float)
    partitions = np.array([x.partition_now for x in logs], dtype=bool)
    runtimes = np.array([x.runtime_ms for x in logs], dtype=float)

    out = {
        "mean_lambda2": float(np.mean(lambda2_vals)) if len(lambda2_vals) else 0.0,
        "partition_count": int(np.sum(partitions)),
        "partition_rate": float(np.mean(partitions)) if len(partitions) else 0.0,
        "runtime_median_ms": float(np.median(runtimes)) if len(runtimes) else 0.0,
        "runtime_iqr_low_ms": float(np.percentile(runtimes, 25)) if len(runtimes) else 0.0,
        "runtime_iqr_high_ms": float(np.percentile(runtimes, 75)) if len(runtimes) else 0.0,
    }
    return out

# ============================================================
# Simple benchmark (runtime vs n)
# ============================================================

def benchmark_runtime_vs_n(
    base_cfg: ACLDConfig,
    sizes: Iterable[int] = (50, 150, 500, 1000),
    repeats: int = 200,
) -> List[Dict[str, float]]:
    rows: List[Dict[str, float]] = []

    for n in sizes:
        cfg = ACLDConfig(**{**base_cfg.__dict__, "n_uavs": int(n)})
        rng = make_rng(cfg.seed + int(n))
        state = SwarmState.random_init(cfg, rng)
        acld = ACLDCore(cfg)

        times_ms = np.zeros(repeats, dtype=float)
        for k in range(repeats):
            t0 = time.perf_counter()
            _ = acld.run_acld(state)
            t1 = time.perf_counter()
            times_ms[k] = (t1 - t0) * 1000.0
            # small random motion to avoid exact cache repetition
            state.step_random_kinematics(cfg, rng)

        row = {
            "n": int(n),
            "pairs": int(n * (n - 1) // 2),
            "median_ms": float(np.median(times_ms)),
            "q1_ms": float(np.percentile(times_ms, 25)),
            "q3_ms": float(np.percentile(times_ms, 75)),
            "mean_ms": float(np.mean(times_ms)),
            "p95_ms": float(np.percentile(times_ms, 95)),
        }
        rows.append(row)

    return rows

# ============================================================
# Optional: one-step evaluation helper for PEW/FP style labels
# ============================================================

def connectivity_components_from_adjacency(A: np.ndarray) -> Tuple[int, np.ndarray]:
    G = csr_matrix(A.astype(np.int8))
    n_comp, labels = connected_components(G, directed=False, return_labels=True)
    return int(n_comp), labels


def edge_deletion_disconnects(A: np.ndarray, edge: Tuple[int, int]) -> bool:
    i, j = edge
    if A[i, j] == 0:
        return False
    A2 = A.copy()
    A2[i, j] = 0
    A2[j, i] = 0
    n_comp_before, _ = connectivity_components_from_adjacency(A)
    n_comp_after, _ = connectivity_components_from_adjacency(A2)
    return n_comp_after > n_comp_before


def approximate_edge_f1_on_predicted_graph(
    A_pred_viable: np.ndarray,
    flagged_edges: List[Tuple[int, int]],
    sample_size: int = 200,
    rng: Optional[np.random.Generator] = None,
) -> Dict[str, float]:
    """
    Approximate edge-level F1 by sampling edges and testing edge deletion connectivity impact.
    """
    if rng is None:
        rng = np.random.default_rng(123)

    n = A_pred_viable.shape[0]
    edges = [(i, j) for i in range(n) for j in range(i + 1, n) if A_pred_viable[i, j] == 1]
    if not edges:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    if len(edges) > sample_size:
        idx = rng.choice(len(edges), size=sample_size, replace=False)
        sampled = [edges[k] for k in idx]
    else:
        sampled = edges

    true_critical = set(e for e in sampled if edge_deletion_disconnects(A_pred_viable, e))
    flagged = set(flagged_edges)

    tp = len(true_critical & flagged)
    fp = len(flagged - true_critical)
    fn = len(true_critical - flagged)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}

# ============================================================
# Paper-style experiment + plotting suite (ACLD full eval)
# ============================================================

@dataclass
class ScenarioSpec:
    name: str
    cfg_overrides: Dict[str, object]
    horizon_s: float = 10.0
    seeds: Tuple[int, ...] = (42, 43, 44, 45, 46)
    ebc_every_k_steps: int = 5  # exact EBC is expensive; evaluate every k-th step


def _clone_cfg(base_cfg: ACLDConfig, **overrides) -> ACLDConfig:
    d = dict(base_cfg.__dict__)
    d.update(overrides)
    return ACLDConfig(**d)


def _ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def _save_rows_csv(path: str, rows: List[Dict[str, object]]) -> None:
    _ensure_dir(os.path.dirname(path) or ".")
    if not rows:
        with open(path, "w", newline="", encoding="utf-8") as f:
            f.write("")
        return
    keys = sorted({k for r in rows for k in r.keys()})
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow(r)


def _fmt_pm(mean_val, std_val, digits: int = 3) -> str:
    """Format as 'mean ± std' with fixed precision; returns '-' if mean is NaN."""
    try:
        mv = float(mean_val)
    except Exception:
        return "-"
    if mv != mv:
        return "-"
    try:
        sv = float(std_val)
    except Exception:
        return f"{mv:.{digits}f}"
    if sv != sv:
        return f"{mv:.{digits}f}"
    return f"{mv:.{digits}f} ± {sv:.{digits}f}"


def _build_manuscript_main_metric_table(agg_rows: List[Dict[str, object]]) -> pd.DataFrame:
    """Build a paper-ready main metrics table (scenario-level) from agg_rows."""
    if not agg_rows:
        return pd.DataFrame()

    order = {
        "nominal_dense": 0,
        "nominal_sparse": 1,
        "nominal_medium": 2,
        "jam_5dB_medium": 3,
        "jam_15dB_medium": 4,
    }
    label = {
        "nominal_dense": "Nominal (Dense)",
        "nominal_sparse": "Nominal (Sparse)",
        "nominal_medium": "Nominal (Medium)",
        "jam_5dB_medium": "Jamming (5 dB)",
        "jam_15dB_medium": "Jamming (15 dB)",
    }

    rows = sorted(agg_rows, key=lambda r: (order.get(str(r.get("scenario")), 999), str(r.get("scenario"))))
    out_rows = []
    for r in rows:
        sc = str(r.get("scenario"))
        out_rows.append({
            "Scenario": label.get(sc, sc),
            "n_episodes": r.get("n_episodes", ""),

            "PEW-F1 (ACLD)": _fmt_pm(r.get("acld_f1_mean", np.nan), r.get("acld_f1_std", np.nan)),
            "PEW-F1 (Tarjan+Guard)": _fmt_pm(r.get("tarjan_guard_f1_mean", np.nan), r.get("tarjan_guard_f1_std", np.nan)),
            "PEW-F1 (Tarjan-Geom)": _fmt_pm(r.get("tarjan_geom_f1_mean", np.nan), r.get("tarjan_geom_f1_std", np.nan)),

            "FPR (ACLD)": _fmt_pm(r.get("acld_fpr_mean", np.nan), r.get("acld_fpr_std", np.nan)),
            "FPR (Tarjan+Guard)": _fmt_pm(r.get("tarjan_guard_fpr_mean", np.nan), r.get("tarjan_guard_fpr_std", np.nan)),
            "FPR (Tarjan-Geom)": _fmt_pm(r.get("tarjan_geom_fpr_mean", np.nan), r.get("tarjan_geom_fpr_std", np.nan)),

            "Edge-F1 (ACLD)": _fmt_pm(r.get("edge_f1_acld_mean_mean", np.nan), r.get("edge_f1_acld_mean_std", np.nan)),
            "Edge-F1 (Tarjan)": _fmt_pm(r.get("edge_f1_tarjan_mean_mean", np.nan), r.get("edge_f1_tarjan_mean_std", np.nan)),
            "Edge-F1 (EBC)": _fmt_pm(r.get("edge_f1_ebc_mean_mean", np.nan), r.get("edge_f1_ebc_mean_std", np.nan)),

            "Runtime median (ms)": _fmt_pm(r.get("runtime_median_ms_mean", np.nan), r.get("runtime_median_ms_std", np.nan)),
            "Runtime p95 (ms)": _fmt_pm(r.get("runtime_p95_ms_mean", np.nan), r.get("runtime_p95_ms_std", np.nan)),
        })

    return pd.DataFrame(out_rows)

def _save_json(path: str, obj: object) -> None:
    _ensure_dir(os.path.dirname(path) or ".")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def _binary_confusion_metrics(y_true: List[int], y_pred: List[int]) -> Dict[str, float]:
    yt = np.array(y_true, dtype=int)
    yp = np.array(y_pred, dtype=int)
    if yt.size == 0:
        return dict(tp=0, fp=0, tn=0, fn=0, precision=0.0, recall=0.0, f1=0.0, fpr=0.0, tpr=0.0, acc=0.0)

    tp = int(np.sum((yt == 1) & (yp == 1)))
    fp = int(np.sum((yt == 0) & (yp == 1)))
    tn = int(np.sum((yt == 0) & (yp == 0)))
    fn = int(np.sum((yt == 1) & (yp == 0)))

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    tpr = recall
    acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0.0

    return dict(tp=tp, fp=fp, tn=tn, fn=fn, precision=precision, recall=recall, f1=f1, fpr=fpr, tpr=tpr, acc=acc)


def _nanmean(vals: List[float]) -> float:
    if not vals:
        return float("nan")
    a = np.array(vals, dtype=float)
    if a.size == 0:
        return float("nan")
    return float(np.nanmean(a))


def _nanstd(vals: List[float]) -> float:
    if not vals:
        return float("nan")
    a = np.array(vals, dtype=float)
    if a.size == 0:
        return float("nan")
    return float(np.nanstd(a))


def build_current_viable_adjacency(acld: ACLDCore, state: SwarmState, with_guard: bool = True) -> np.ndarray:
    """
    Current snapshot adjacency on V' (UAVs + optional GS), using theta_plus and optionally p_min guard.
    """
    cfg = acld.cfg
    n_uav = cfg.n_uavs
    n_total = cfg.n_total
    sidx = cfg.gs_index

    d_cur_uav = acld._pairwise_uav_dist(state.pos)
    _, _, theta_plus_uav = acld.adaptive_threshold_uav_uav(state, d_cur_uav)
    s_cur_uav = acld.success_probability_uav_uav(d_cur_uav, pos_for_jammer=state.pos)

    A = np.zeros((n_total, n_total), dtype=np.uint8)

    mask_uav = (d_cur_uav <= theta_plus_uav)
    if with_guard:
        mask_uav &= (s_cur_uav >= cfg.p_min)
    np.fill_diagonal(mask_uav, False)
    A[:n_uav, :n_uav] = mask_uav.astype(np.uint8)

    if cfg.include_ground_station:
        d_cur_gs = acld._uav_gs_dist(state.pos)
        _, _, theta_plus_gs = acld.adaptive_threshold_uav_gs(state, d_cur_uav)
        s_cur_gs = acld.success_probability_uav_gs(d_cur_gs, state.pos)

        for i in range(n_uav):
            ok = bool(d_cur_gs[i] <= theta_plus_gs[i])
            if with_guard:
                ok = ok and bool(s_cur_gs[i] >= cfg.p_min)
            if ok:
                A[i, sidx] = 1
                A[sidx, i] = 1

    return A


def binary_lambda2_from_adjacency(A: np.ndarray) -> float:
    D = np.diag(np.sum(A.astype(float), axis=1))
    L = D - A.astype(float)
    return ACLDCore.algebraic_connectivity_lambda2(L)


def topk_ebc_edges(A: np.ndarray, k: int) -> List[Tuple[int, int]]:
    if k <= 0:
        return []
    eb = ACLDCore.edge_betweenness_brandes_unweighted(A)
    if not eb:
        return []
    ranked = sorted(eb.items(), key=lambda kv: kv[1], reverse=True)
    return [e for e, _ in ranked[:k]]


def _prf_from_edge_counts(tp: int, fp: int, fn: int) -> Dict[str, float]:
    """Precision/recall/F1 from exact edge-level counts.

    When tp=fp=fn=0 there is no localization support in that unit, so P/R/F1
    are undefined (NaN), not zero. This prevents a correct no-bridge/no-alarm
    step from being counted as localization failure.
    """
    tp, fp, fn = int(tp), int(fp), int(fn)
    if (tp + fp + fn) == 0:
        return {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")}
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else float("nan")
    return {"precision": float(precision), "recall": float(recall), "f1": float(f1)}


def _edge_f1_exact_vs_bridges(A_target: np.ndarray, flagged_edges: List[Tuple[int, int]]) -> Dict[str, float]:
    """Exact bridge-localization counts on the realized next viable graph.

    Ground truth is the exact Tarjan bridge set of `A_target`. Every valid
    predicted node-pair is retained. Therefore a predicted pair absent from the
    realized next graph is a false positive rather than being silently removed.
    """
    true_critical = {stable_edge(e) for e in ACLDCore.tarjan_bridges(A_target)}
    flagged = set()
    n = A_target.shape[0]
    for i, j in flagged_edges:
        i, j = int(i), int(j)
        if 0 <= i < n and 0 <= j < n and i != j:
            flagged.add(stable_edge((i, j)))

    tp = len(true_critical & flagged)
    fp = len(flagged - true_critical)
    fn = len(true_critical - flagged)
    prf = _prf_from_edge_counts(tp, fp, fn)
    return {**prf, "tp": int(tp), "fp": int(fp), "fn": int(fn)}


def run_evaluated_episode(
    cfg: ACLDConfig,
    horizon_s: float = 10.0,
    seed: Optional[int] = None,
    ebc_every_k_steps: int = 5,
    compute_exact_edge_f1_for_n_leq: int = 220,
) -> Tuple[List[Dict[str, object]], Dict[str, object]]:
    """
    Runs one episode and evaluates one-step-ahead prediction/alerts vs actual next snapshot.
    Returns:
      step_rows: list of per-step dicts
      episode_summary: dict
    """
    rng = make_rng(cfg.seed if seed is None else seed)
    state = SwarmState.random_init(cfg, rng)
    acld = ACLDCore(cfg)

    T = int(round(horizon_s / cfg.dt_sim))
    step_rows: List[Dict[str, object]] = []

    # For confusion metrics
    y_true_partition_next: List[int] = []
    yhat_acld: List[int] = []
    yhat_tarjan_guard: List[int] = []
    yhat_tarjan_geom: List[int] = []

    e1_acld_list: List[float] = []
    e1_tarjan_list: List[float] = []
    e1_ebc_list: List[float] = []

    t_now = 0.0
    for t_idx in range(max(0, T - 1)):
        t0 = time.perf_counter()
        res = acld.run_acld(state)
        t1 = time.perf_counter()
        runtime_ms = (t1 - t0) * 1000.0

        A_now_viable = build_current_viable_adjacency(acld, state, with_guard=True)
        A_now_geom = build_current_viable_adjacency(acld, state, with_guard=False)

        # Baselines at t
        tarjan_guard_edges = ACLDCore.tarjan_bridges(A_now_viable)
        tarjan_geom_edges = ACLDCore.tarjan_bridges(A_now_geom)

        # EBC baseline (subsampled in time due cost)
        do_ebc = (ebc_every_k_steps <= 1) or (t_idx % ebc_every_k_steps == 0)
        ebc_edges = []
        if do_ebc:
            k_ebc = max(1, len(res.risky_inter)) if len(res.risky_inter) > 0 else 1
            ebc_edges = topk_ebc_edges(A_now_viable, k=k_ebc)

        # Advance one step to obtain actual next label/graph
        next_state = state.copy()
        next_state.step_random_kinematics(cfg, rng)

        A_next_viable = build_current_viable_adjacency(acld, next_state, with_guard=True)
        n_comp_next, _ = connectivity_components_from_adjacency(A_next_viable)
        lambda2_next_bin = binary_lambda2_from_adjacency(A_next_viable)
        lambda2_next_weighted = acld._build_current_weighted_graph(next_state)[2]
        lambda2_next_weighted_val = ACLDCore.algebraic_connectivity_lambda2(lambda2_next_weighted)

        # One-step partition label (binary)
        # Positive if actual next viable graph is disconnected (components > 1)
        y_true = 1 if n_comp_next > 1 else 0

        # ACLD-v2 alert rule: predicted viable fragmentation / lambda2 collapse,
        # optionally supported by forecasted bridge structure.
        acld_alert = int(
            (res.n_components_pred_viable > 1)
            or (res.lambda2_pred_viable < cfg.lambda2_partition_threshold)
            or ((len(res.predicted_viable_bridges) > 0) and (len(res.risky_inter) > 0))
        )

        tarjan_guard_alert = int(len(tarjan_guard_edges) > 0)
        tarjan_geom_alert = int(len(tarjan_geom_edges) > 0)

        y_true_partition_next.append(y_true)
        yhat_acld.append(acld_alert)
        yhat_tarjan_guard.append(tarjan_guard_alert)
        yhat_tarjan_geom.append(tarjan_geom_alert)

        # Edge-level F1 vs next-step bridges (exact for moderate n, approximate for larger n)
        target_flagged_acld = list(res.selected_critical_edges)
        if cfg.n_total <= compute_exact_edge_f1_for_n_leq:
            f1_acld = _edge_f1_exact_vs_bridges(A_next_viable, target_flagged_acld)["f1"]
            f1_tarjan = _edge_f1_exact_vs_bridges(A_next_viable, tarjan_guard_edges)["f1"]
            if do_ebc:
                f1_ebc = _edge_f1_exact_vs_bridges(A_next_viable, ebc_edges)["f1"]
            else:
                f1_ebc = np.nan
        else:
            f1_acld = approximate_edge_f1_on_predicted_graph(A_next_viable, target_flagged_acld, sample_size=200, rng=rng)["f1"]
            f1_tarjan = approximate_edge_f1_on_predicted_graph(A_next_viable, tarjan_guard_edges, sample_size=200, rng=rng)["f1"]
            if do_ebc:
                f1_ebc = approximate_edge_f1_on_predicted_graph(A_next_viable, ebc_edges, sample_size=200, rng=rng)["f1"]
            else:
                f1_ebc = np.nan

        e1_acld_list.append(float(f1_acld))
        e1_tarjan_list.append(float(f1_tarjan))
        if do_ebc:
            e1_ebc_list.append(float(f1_ebc))

        step_rows.append(
            {
                "t": t_now,
                "step_idx": t_idx,
                "runtime_ms": runtime_ms,
                "lambda2_current_weighted": float(res.lambda2_current),
                "lambda2_pred_viable_bin": float(res.lambda2_pred_viable),
                "lambda2_next_viable_bin": float(lambda2_next_bin),
                "lambda2_next_weighted": float(lambda2_next_weighted_val),
                "n_components_pred_guar": int(len(res.components)),
                "n_components_next_viable": int(n_comp_next),
                "n_risky_inter": int(len(res.risky_inter)),
                "n_risky_intra": int(len(res.risky_intra)),
                "n_risky_s_uav": int(len(res.risky_s_uav)),
                "n_risky_uav_uav": int(len(res.risky_uav_uav)),
                "tarjan_guard_bridges": int(len(tarjan_guard_edges)),
                "tarjan_geom_bridges": int(len(tarjan_geom_edges)),
                "ebc_k": int(len(ebc_edges)),
                "label_partition_next": int(y_true),
                "alert_acld": int(acld_alert),
                "alert_tarjan_guard": int(tarjan_guard_alert),
                "alert_tarjan_geom": int(tarjan_geom_alert),
                "edge_f1_acld_vs_next": float(f1_acld),
                "edge_f1_tarjan_vs_next": float(f1_tarjan),
                "edge_f1_ebc_vs_next": float(f1_ebc) if do_ebc else np.nan,
            }
        )

        # move to next state
        state = next_state
        t_now += cfg.dt_sim

    m_acld = _binary_confusion_metrics(y_true_partition_next, yhat_acld)
    m_tg = _binary_confusion_metrics(y_true_partition_next, yhat_tarjan_guard)
    m_tgeom = _binary_confusion_metrics(y_true_partition_next, yhat_tarjan_geom)

    runtimes = [r["runtime_ms"] for r in step_rows]
    summary = {
        "n_steps_eval": int(len(step_rows)),
        "partition_next_rate": float(np.mean(y_true_partition_next)) if y_true_partition_next else 0.0,
        "runtime_median_ms": float(np.median(runtimes)) if runtimes else 0.0,
        "runtime_p95_ms": float(np.percentile(runtimes, 95)) if runtimes else 0.0,
        "mean_lambda2_current_weighted": float(np.mean([r["lambda2_current_weighted"] for r in step_rows])) if step_rows else 0.0,

        # ACLD confusion
        "acld_precision": m_acld["precision"],
        "acld_recall": m_acld["recall"],
        "acld_f1": m_acld["f1"],
        "acld_fpr": m_acld["fpr"],
        "acld_acc": m_acld["acc"],

        # Tarjan guard
        "tarjan_guard_precision": m_tg["precision"],
        "tarjan_guard_recall": m_tg["recall"],
        "tarjan_guard_f1": m_tg["f1"],
        "tarjan_guard_fpr": m_tg["fpr"],
        "tarjan_guard_acc": m_tg["acc"],

        # Tarjan geom-only
        "tarjan_geom_precision": m_tgeom["precision"],
        "tarjan_geom_recall": m_tgeom["recall"],
        "tarjan_geom_f1": m_tgeom["f1"],
        "tarjan_geom_fpr": m_tgeom["fpr"],
        "tarjan_geom_acc": m_tgeom["acc"],

        # Edge F1
        "edge_f1_acld_mean": _nanmean(e1_acld_list),
        "edge_f1_tarjan_mean": _nanmean(e1_tarjan_list),
        "edge_f1_ebc_mean": _nanmean(e1_ebc_list),
    }
    return step_rows, summary


def run_scenario_mc(
    base_cfg: ACLDConfig,
    spec: ScenarioSpec,
) -> Tuple[List[Dict[str, object]], List[Dict[str, object]], Dict[str, object]]:
    """
    Returns:
      scenario_episode_rows (one row per seed),
      scenario_step_rows (all steps, tagged),
      aggregated_summary
    """
    cfg = _clone_cfg(base_cfg, **spec.cfg_overrides)

    episode_rows: List[Dict[str, object]] = []
    all_step_rows: List[Dict[str, object]] = []

    for seed in spec.seeds:
        step_rows, ep_summary = run_evaluated_episode(
            cfg=cfg,
            horizon_s=spec.horizon_s,
            seed=seed,
            ebc_every_k_steps=spec.ebc_every_k_steps,
        )
        ep_row = {
            "scenario": spec.name,
            "seed": seed,
            "n_uavs": cfg.n_uavs,
            "density_scale_mode": cfg.density_scale_mode,
            "jnr_db": cfg.jnr_db if cfg.jnr_db is not None else "NoJam",
            **ep_summary,
        }
        episode_rows.append(ep_row)

        for r in step_rows:
            rr = dict(r)
            rr["scenario"] = spec.name
            rr["seed"] = seed
            rr["n_uavs"] = cfg.n_uavs
            rr["density_scale_mode"] = cfg.density_scale_mode
            rr["jnr_db"] = cfg.jnr_db if cfg.jnr_db is not None else "NoJam"
            all_step_rows.append(rr)

    # Aggregate across seeds
    def _agg(key: str) -> Tuple[float, float]:
        vals = [float(r[key]) for r in episode_rows if key in r and r[key] == r[key]]
        if not vals:
            return float("nan"), float("nan")
        return float(np.mean(vals)), float(np.std(vals))

    agg = {
        "scenario": spec.name,
        "n_episodes": len(episode_rows),
        "n_uavs": cfg.n_uavs,
        "density_scale_mode": cfg.density_scale_mode,
        "jnr_db": cfg.jnr_db if cfg.jnr_db is not None else "NoJam",
    }

    keys_to_aggregate = [
        "partition_next_rate",
        "runtime_median_ms",
        "runtime_p95_ms",
        "mean_lambda2_current_weighted",
        "acld_precision", "acld_recall", "acld_f1", "acld_fpr", "acld_acc",
        "tarjan_guard_precision", "tarjan_guard_recall", "tarjan_guard_f1", "tarjan_guard_fpr", "tarjan_guard_acc",
        "tarjan_geom_precision", "tarjan_geom_recall", "tarjan_geom_f1", "tarjan_geom_fpr", "tarjan_geom_acc",
        "edge_f1_acld_mean", "edge_f1_tarjan_mean", "edge_f1_ebc_mean",
    ]

    for k in keys_to_aggregate:
        mu, sd = _agg(k)
        agg[f"{k}_mean"] = mu
        agg[f"{k}_std"] = sd

    return episode_rows, all_step_rows, agg


# --------------------------
# Plotting helpers
# --------------------------

def _plot_timeseries_representative(step_rows: List[Dict[str, object]], outdir: str, prefix: str = "fig") -> None:
    if not step_rows:
        return

    t = np.array([r["t"] for r in step_rows], dtype=float)
    lam = np.array([r["lambda2_current_weighted"] for r in step_rows], dtype=float)
    lam_pred = np.array([r["lambda2_pred_viable_bin"] for r in step_rows], dtype=float)
    lam_next = np.array([r["lambda2_next_viable_bin"] for r in step_rows], dtype=float)
    r_inter = np.array([r["n_risky_inter"] for r in step_rows], dtype=float)
    r_intra = np.array([r["n_risky_intra"] for r in step_rows], dtype=float)
    r_s = np.array([r["n_risky_s_uav"] for r in step_rows], dtype=float)
    r_uu = np.array([r["n_risky_uav_uav"] for r in step_rows], dtype=float)
    comp_pred = np.array([r["n_components_pred_guar"] for r in step_rows], dtype=float)
    comp_next = np.array([r["n_components_next_viable"] for r in step_rows], dtype=float)
    rt = np.array([r["runtime_ms"] for r in step_rows], dtype=float)
    y = np.array([r["label_partition_next"] for r in step_rows], dtype=float)

    # Figure 1: lambda2 trends
    plt.figure(figsize=(10, 4.8))
    plt.plot(t, lam, label="λ2 current (weighted)")
    plt.plot(t, lam_pred, label="λ2 pred viable (binary)")
    plt.plot(t, lam_next, label="λ2 next viable (binary)")
    plt.xlabel("Time (s)")
    plt.ylabel("Connectivity metric")
    plt.title("Connectivity traces over time")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{prefix}_01_lambda2_timeseries.png"), dpi=180)
    plt.close()

    # Figure 2: risky link counts
    plt.figure(figsize=(10, 4.8))
    plt.plot(t, r_inter, label="risky inter")
    plt.plot(t, r_intra, label="risky intra")
    plt.plot(t, r_s, label="risky S-UAV")
    plt.plot(t, r_uu, label="risky UAV-UAV")
    plt.xlabel("Time (s)")
    plt.ylabel("Count")
    plt.title("Risky link counts over time")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{prefix}_02_risky_link_counts.png"), dpi=180)
    plt.close()

    # Figure 3: component counts + partition label
    plt.figure(figsize=(10, 4.8))
    plt.plot(t, comp_pred, label="pred guaranteed components")
    plt.plot(t, comp_next, label="next viable components")
    plt.step(t, y, where="post", label="partition label (next)", alpha=0.8)
    plt.xlabel("Time (s)")
    plt.ylabel("Count / label")
    plt.title("Predicted vs actual next-step partition behavior")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{prefix}_03_components_partition_label.png"), dpi=180)
    plt.close()

    # Figure 4: runtime
    plt.figure(figsize=(10, 4.8))
    plt.plot(t, rt)
    plt.xlabel("Time (s)")
    plt.ylabel("Runtime (ms)")
    plt.title("ACLD per-step runtime")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{prefix}_04_runtime_timeseries.png"), dpi=180)
    plt.close()


def _plot_runtime_scaling(bench_rows: List[Dict[str, float]], outdir: str, prefix: str = "fig") -> None:
    if not bench_rows:
        return
    n = np.array([r["n"] for r in bench_rows], dtype=float)
    med = np.array([r["median_ms"] for r in bench_rows], dtype=float)
    q1 = np.array([r["q1_ms"] for r in bench_rows], dtype=float)
    q3 = np.array([r["q3_ms"] for r in bench_rows], dtype=float)
    p95 = np.array([r["p95_ms"] for r in bench_rows], dtype=float)

    plt.figure(figsize=(8, 5))
    plt.plot(n, med, marker="o", label="Median")
    plt.fill_between(n, q1, q3, alpha=0.2, label="IQR")
    plt.plot(n, p95, marker="s", linestyle="--", label="P95")
    plt.xlabel("Number of UAVs")
    plt.ylabel("Runtime (ms)")
    plt.title("Runtime scaling vs swarm size")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{prefix}_05_runtime_scaling.png"), dpi=180)
    plt.close()


def _plot_scenario_bars(agg_rows: List[Dict[str, object]], outdir: str, prefix: str = "fig") -> None:
    if not agg_rows:
        return

    order = {
        "nominal_dense": 0,
        "nominal_sparse": 1,
        "nominal_medium": 2,
        "jam_5dB_medium": 3,
        "jam_15dB_medium": 4,
    }
    label = {
        "nominal_dense": "Nominal (Dense)",
        "nominal_sparse": "Nominal (Sparse)",
        "nominal_medium": "Nominal (Medium)",
        "jam_5dB_medium": "Jamming (5 dB)",
        "jam_15dB_medium": "Jamming (15 dB)",
    }

    rows = sorted(agg_rows, key=lambda r: (order.get(str(r.get("scenario")), 999), str(r.get("scenario"))))
    names = [label.get(str(r.get("scenario")), str(r.get("scenario"))) for r in rows]
    x = np.arange(len(names), dtype=float)
    w = 0.23

    def arr(key):
        return np.array([float(r.get(key, np.nan)) for r in rows], dtype=float)

    # Means + stds
    acld_f1, acld_f1_sd = arr("acld_f1_mean"), arr("acld_f1_std")
    tg_f1, tg_f1_sd = arr("tarjan_guard_f1_mean"), arr("tarjan_guard_f1_std")
    tgeom_f1, tgeom_f1_sd = arr("tarjan_geom_f1_mean"), arr("tarjan_geom_f1_std")

    plt.figure(figsize=(max(9, 1.25 * len(names)), 5.2))
    plt.bar(x - w, acld_f1, yerr=acld_f1_sd, capsize=3, width=w, label="ACLD-v2")
    plt.bar(x, tg_f1, yerr=tg_f1_sd, capsize=3, width=w, label="Tarjan+guard")
    plt.bar(x + w, tgeom_f1, yerr=tgeom_f1_sd, capsize=3, width=w, label="Tarjan geom-only")
    plt.xticks(x, names, rotation=25, ha="right")
    plt.ylabel("PEW-F1")
    plt.title("PEW-F1 across scenarios")
    plt.legend()
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{prefix}_06_pew_f1_scenarios.png"), dpi=180)
    plt.close()

    # FPR
    acld_fpr, acld_fpr_sd = arr("acld_fpr_mean"), arr("acld_fpr_std")
    tg_fpr, tg_fpr_sd = arr("tarjan_guard_fpr_mean"), arr("tarjan_guard_fpr_std")
    tgeom_fpr, tgeom_fpr_sd = arr("tarjan_geom_fpr_mean"), arr("tarjan_geom_fpr_std")

    plt.figure(figsize=(max(9, 1.25 * len(names)), 5.2))
    plt.bar(x - w, acld_fpr, yerr=acld_fpr_sd, capsize=3, width=w, label="ACLD")
    plt.bar(x, tg_fpr, yerr=tg_fpr_sd, capsize=3, width=w, label="Tarjan+guard")
    plt.bar(x + w, tgeom_fpr, yerr=tgeom_fpr_sd, capsize=3, width=w, label="Tarjan geom-only")
    plt.xticks(x, names, rotation=25, ha="right")
    plt.ylabel("False Positive Rate")
    plt.title("False-positive behavior across scenarios")
    plt.legend()
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{prefix}_07_fpr_scenarios.png"), dpi=180)
    plt.close()

    # Edge-F1
    acld_e, acld_e_sd = arr("edge_f1_acld_mean_mean"), arr("edge_f1_acld_mean_std")
    tg_e, tg_e_sd = arr("edge_f1_tarjan_mean_mean"), arr("edge_f1_tarjan_mean_std")
    ebc_e, ebc_e_sd = arr("edge_f1_ebc_mean_mean"), arr("edge_f1_ebc_mean_std")

    plt.figure(figsize=(max(9, 1.25 * len(names)), 5.2))
    plt.bar(x - w, acld_e, yerr=acld_e_sd, capsize=3, width=w, label="ACLD")
    plt.bar(x, tg_e, yerr=tg_e_sd, capsize=3, width=w, label="Tarjan+guard")
    plt.bar(x + w, ebc_e, yerr=ebc_e_sd, capsize=3, width=w, label="EBC@t top-k")
    plt.xticks(x, names, rotation=25, ha="right")
    plt.ylabel("Edge-level F1 (vs next bridges)")
    plt.title("Edge-level critical-link prediction quality")
    plt.legend()
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{prefix}_08_edge_f1_scenarios.png"), dpi=180)
    plt.close()

def _plot_runtime_boxplot_per_scenario(episode_rows: List[Dict[str, object]], outdir: str, prefix: str = "fig") -> None:
    if not episode_rows:
        return
    by_s: Dict[str, List[float]] = {}
    for r in episode_rows:
        by_s.setdefault(str(r["scenario"]), []).append(float(r["runtime_median_ms"]))
    names = list(by_s.keys())
    data = [by_s[k] for k in names]

    plt.figure(figsize=(max(8, 1.1 * len(names)), 5))
    plt.boxplot(data, labels=names, showmeans=True)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("Episode median runtime (ms)")
    plt.title("Runtime distribution across scenarios/seeds")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{prefix}_09_runtime_boxplot_scenarios.png"), dpi=180)
    plt.close()


def _plot_jamming_sweep(jam_rows: List[Dict[str, object]], outdir: str, prefix: str = "fig") -> None:
    if not jam_rows:
        return

    def _jnr_sort_val(x):
        if x is None:
            return -1e9
        sx = str(x).strip()
        if sx.lower() in {"noj am", "nojam", "none", "nan"}:
            return -1e9
        try:
            return float(x)
        except Exception:
            return 1e9

    rows = sorted(jam_rows, key=lambda r: _jnr_sort_val(r.get("jnr_db")))

    x = np.arange(len(rows), dtype=float)
    labels = []
    for r in rows:
        v = r.get("jnr_db")
        if v is None or str(v).strip().lower() in {"nojam", "none", "nan"}:
            labels.append("NoJam")
        else:
            try:
                labels.append(f"{float(v):g}")
            except Exception:
                labels.append(str(v))

    def arr(key):
        return np.array([float(r.get(key, np.nan)) for r in rows], dtype=float)

    acld_f1 = arr("acld_f1_mean")
    tg_f1 = arr("tarjan_guard_f1_mean")
    tgeom_f1 = arr("tarjan_geom_f1_mean")

    acld_fpr = arr("acld_fpr_mean")
    tg_fpr = arr("tarjan_guard_fpr_mean")
    tgeom_fpr = arr("tarjan_geom_fpr_mean")

    plt.figure(figsize=(10, 5))
    plt.plot(x, acld_f1, marker="o", label="ACLD-v2 PEW-F1")
    plt.plot(x, tg_f1, marker="s", label="Tarjan+guard PEW-F1")
    plt.plot(x, tgeom_f1, marker="^", label="Tarjan geom-only PEW-F1")

    plt.plot(x, acld_fpr, marker="o", linestyle="--", label="ACLD-v2 FPR")
    plt.plot(x, tg_fpr, marker="s", linestyle="--", label="Tarjan+guard FPR")
    plt.plot(x, tgeom_fpr, marker="^", linestyle="--", label="Tarjan geom-only FPR")

    plt.xticks(x, labels)
    plt.xlabel("JNR (dB)")
    plt.ylabel("Metric value")
    plt.title("Jamming sweep (PEW-F1 / FPR)")
    plt.legend(fontsize=8, ncol=2)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{prefix}_10_jamming_sweep.png"), dpi=180)
    plt.close()

def _plot_density_sweep(density_rows: List[Dict[str, object]], outdir: str, prefix: str = "fig") -> None:
    if not density_rows:
        return
    order = {"sparse": 0, "medium": 1, "dense": 2}
    rows = sorted(density_rows, key=lambda r: order.get(str(r.get("density_scale_mode")), 999))
    label_map = {"sparse": "Sparse", "medium": "Medium", "dense": "Dense"}
    names = [label_map.get(str(r.get("density_scale_mode")), str(r.get("density_scale_mode"))) for r in rows]
    x = np.arange(len(names), dtype=float)

    part = np.array([float(r.get("partition_next_rate_mean", np.nan)) for r in rows], dtype=float)
    acld_f1 = np.array([float(r.get("acld_f1_mean", np.nan)) for r in rows], dtype=float)
    fpr = np.array([float(r.get("acld_fpr_mean", np.nan)) for r in rows], dtype=float)

    plt.figure(figsize=(8.5, 5))
    plt.plot(x, part, marker="o", label="partition-next rate")
    plt.plot(x, acld_f1, marker="s", label="ACLD-v2 PEW-F1")
    plt.plot(x, fpr, marker="^", label="ACLD-v2 FPR")
    plt.xticks(x, names)
    plt.xlabel("Density")
    plt.ylabel("Metric value")
    plt.title("Density sweep")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{prefix}_11_density_sweep.png"), dpi=180)
    plt.close()

def _plot_ablation(ablation_rows: List[Dict[str, object]], outdir: str, prefix: str = "fig") -> None:
    if not ablation_rows:
        return

    order_map = {
        "ablation_full": 0,
        "ablation_no_alpha": 1,
        "ablation_no_beta": 2,
        "ablation_no_gamma": 3,
        "ablation_no_delta": 4,
        "ablation_no_zeta": 5,
        "ablation_fixed_cthr": 6,
    }
    label_map = {
        "ablation_full": "Full",
        "ablation_no_alpha": "w/o α",
        "ablation_no_beta": "w/o β",
        "ablation_no_gamma": "w/o γ",
        "ablation_no_delta": "w/o δ",
        "ablation_no_zeta": "w/o ζ",
        "ablation_fixed_cthr": "Fixed c_thr",
    }

    rows = sorted(ablation_rows, key=lambda r: order_map.get(str(r.get("scenario")), 999))
    names = [label_map.get(str(r.get("scenario")), str(r.get("scenario"))) for r in rows]
    x = np.arange(len(names), dtype=float)

    acld_f1 = np.array([float(r.get("acld_f1_mean", np.nan)) for r in rows], dtype=float)
    acld_fpr = np.array([float(r.get("acld_fpr_mean", np.nan)) for r in rows], dtype=float)
    edge_f1 = np.array([float(r.get("edge_f1_acld_mean_mean", np.nan)) for r in rows], dtype=float)

    plt.figure(figsize=(max(9, 1.2 * len(names)), 5))
    plt.plot(x, acld_f1, marker="o", label="ACLD-v2 PEW-F1")
    plt.plot(x, acld_fpr, marker="s", label="ACLD-v2 FPR")
    plt.plot(x, edge_f1, marker="^", label="ACLD-v2 Edge-F1")
    plt.xticks(x, names, rotation=25, ha="right")
    plt.ylabel("Metric")
    plt.title("Ablation study (adaptive terms)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{prefix}_12_ablation.png"), dpi=180)
    plt.close()

def run_full_paper_suite(
    outdir: str = "acld_paper_outputs",
    base_cfg: Optional[ACLDConfig] = None,
    fast_mode: bool = False,
) -> Dict[str, object]:
    """
    Generates:
      - representative episode timeseries plots
      - scenario MC summaries + plots
      - runtime scaling benchmark
      - jamming sweep / density sweep
      - ablation study
      - CSV + JSON outputs
    """
    _ensure_dir(outdir)
    figdir = os.path.join(outdir, "figures")
    tabdir = os.path.join(outdir, "tables")
    rawdir = os.path.join(outdir, "raw")
    for d in (figdir, tabdir, rawdir):
        _ensure_dir(d)

    if base_cfg is None:
        base_cfg = ACLDConfig(
            n_uavs=150,
            include_ground_station=True,
            density_scale_mode="medium",
            jnr_db=None,
            seed=42,
        )

    # Speed presets
    if fast_mode:
        rep_horizon = 5.0
        mc_horizon = 6.0
        seeds_small = (42, 43, 44)
        repeats_bench = 20
        bench_sizes = (50, 150, 300)
    else:
        rep_horizon = 15.0
        mc_horizon = 12.0
        seeds_small = (42, 43, 44, 45, 46)
        repeats_bench = 60
        bench_sizes = (50, 150, 300, 500)

    # 1) Representative episode (timeseries figures)
    rep_spec = ScenarioSpec(
        name="representative_nominal",
        cfg_overrides=dict(jnr_db=None, density_scale_mode="medium"),
        horizon_s=rep_horizon,
        seeds=(base_cfg.seed,),
        ebc_every_k_steps=5,
    )
    rep_ep_rows, rep_step_rows, rep_agg = run_scenario_mc(base_cfg, rep_spec)
    _save_rows_csv(os.path.join(rawdir, "representative_episode_summary.csv"), rep_ep_rows)
    _save_rows_csv(os.path.join(rawdir, "representative_episode_steps.csv"), rep_step_rows)
    _save_json(os.path.join(rawdir, "representative_episode_agg.json"), rep_agg)
    # Only first seed representative rows
    rep_seed_rows = [r for r in rep_step_rows if int(r["seed"]) == int(base_cfg.seed)]
    rep_seed_rows = sorted(rep_seed_rows, key=lambda r: float(r["t"]))
    _plot_timeseries_representative(rep_seed_rows, figdir)

    # 2) Main scenario set (nominal / jamming / density)
    scenarios = [
        ScenarioSpec("nominal_medium", dict(jnr_db=None, density_scale_mode="medium"), horizon_s=mc_horizon, seeds=seeds_small),
        ScenarioSpec("jam_5dB_medium", dict(jnr_db=5.0, density_scale_mode="medium"), horizon_s=mc_horizon, seeds=seeds_small),
        ScenarioSpec("jam_15dB_medium", dict(jnr_db=15.0, density_scale_mode="medium"), horizon_s=mc_horizon, seeds=seeds_small),
        ScenarioSpec("nominal_sparse", dict(jnr_db=None, density_scale_mode="sparse"), horizon_s=mc_horizon, seeds=seeds_small),
        ScenarioSpec("nominal_dense", dict(jnr_db=None, density_scale_mode="dense"), horizon_s=mc_horizon, seeds=seeds_small),
    ]

    all_episode_rows: List[Dict[str, object]] = []
    all_step_rows: List[Dict[str, object]] = []
    agg_rows: List[Dict[str, object]] = []

    for spec in scenarios:
        ep_rows, st_rows, agg = run_scenario_mc(base_cfg, spec)
        all_episode_rows.extend(ep_rows)
        all_step_rows.extend(st_rows)
        agg_rows.append(agg)

    _save_rows_csv(os.path.join(tabdir, "scenario_episode_metrics.csv"), all_episode_rows)
    _save_rows_csv(os.path.join(rawdir, "scenario_step_metrics.csv"), all_step_rows)
    _save_rows_csv(os.path.join(tabdir, "scenario_aggregated_summary.csv"), agg_rows)

    # Paper-ready main metric table (mean ± std) for direct manuscript use
    main_tbl = _build_manuscript_main_metric_table(agg_rows)
    if not main_tbl.empty:
        main_tbl_csv = os.path.join(tabdir, "main_metric_table_manuscript.csv")
        main_tbl_tex = os.path.join(tabdir, "main_metric_table_manuscript.tex")
        main_tbl.to_csv(main_tbl_csv, index=False)
        try:
            main_tbl.to_latex(main_tbl_tex, index=False, escape=False)
        except Exception as e:
            print(f"[WARN] LaTeX export failed for main metric table: {e}")

    _plot_scenario_bars(agg_rows, figdir)
    _plot_runtime_boxplot_per_scenario(all_episode_rows, figdir)

    # 3) Runtime scaling benchmark
    bench_rows = benchmark_runtime_vs_n(base_cfg, sizes=bench_sizes, repeats=repeats_bench)
    _save_rows_csv(os.path.join(tabdir, "runtime_scaling.csv"), bench_rows)
    _plot_runtime_scaling(bench_rows, figdir)

    # 4) Jamming sweep
    jam_specs = [
        ScenarioSpec(f"jnr_{'nominal' if j is None else str(int(j))+'dB'}",
                     dict(jnr_db=j, density_scale_mode="medium"),
                     horizon_s=mc_horizon, seeds=seeds_small)
        for j in (None, 0.0, 5.0, 10.0, 15.0, 20.0)
    ]
    jam_agg_rows = []
    for spec in jam_specs:
        _, _, agg = run_scenario_mc(base_cfg, spec)
        jam_agg_rows.append(agg)
    _save_rows_csv(os.path.join(tabdir, "jamming_sweep_summary.csv"), jam_agg_rows)
    _plot_jamming_sweep(jam_agg_rows, figdir)

    # 5) Density sweep
    density_specs = [
        ScenarioSpec(f"density_{mode}", dict(jnr_db=None, density_scale_mode=mode),
                     horizon_s=mc_horizon, seeds=seeds_small)
        for mode in ("sparse", "medium", "dense")
    ]
    density_agg_rows = []
    for spec in density_specs:
        _, _, agg = run_scenario_mc(base_cfg, spec)
        density_agg_rows.append(agg)
    _save_rows_csv(os.path.join(tabdir, "density_sweep_summary.csv"), density_agg_rows)
    _plot_density_sweep(density_agg_rows, figdir)

    # 6) Ablation study (adaptive terms)
    ablation_specs = [
        ScenarioSpec("ablation_full", dict(jnr_db=None, density_scale_mode="medium"), horizon_s=mc_horizon, seeds=seeds_small),
        ScenarioSpec("ablation_no_alpha", dict(alpha=0.0, jnr_db=None, density_scale_mode="medium"), horizon_s=mc_horizon, seeds=seeds_small),
        ScenarioSpec("ablation_no_beta", dict(beta=0.0, jnr_db=None, density_scale_mode="medium"), horizon_s=mc_horizon, seeds=seeds_small),
        ScenarioSpec("ablation_no_gamma", dict(gamma=0.0, jnr_db=None, density_scale_mode="medium"), horizon_s=mc_horizon, seeds=seeds_small),
        ScenarioSpec("ablation_no_delta", dict(delta=0.0, jnr_db=None, density_scale_mode="medium"), horizon_s=mc_horizon, seeds=seeds_small),
        ScenarioSpec("ablation_no_zeta", dict(zeta=0.0, jnr_db=None, density_scale_mode="medium"), horizon_s=mc_horizon, seeds=seeds_small),
        ScenarioSpec("ablation_fixed_cthr", dict(alpha=0.0, beta=0.0, gamma=0.0, delta=0.0, zeta=0.0,
                                                 jnr_db=None, density_scale_mode="medium"),
                     horizon_s=mc_horizon, seeds=seeds_small),
    ]
    ablation_agg_rows = []
    for spec in ablation_specs:
        _, _, agg = run_scenario_mc(base_cfg, spec)
        ablation_agg_rows.append(agg)
    _save_rows_csv(os.path.join(tabdir, "ablation_summary.csv"), ablation_agg_rows)
    _plot_ablation(ablation_agg_rows, figdir)

    # 7) Compact summary JSON
    summary = {
        "outdir": outdir,
        "fast_mode": fast_mode,
        "base_cfg": {
            "n_uavs": base_cfg.n_uavs,
            "include_ground_station": base_cfg.include_ground_station,
            "density_scale_mode": base_cfg.density_scale_mode,
            "jnr_db": base_cfg.jnr_db,
            "seed": base_cfg.seed,
            "p_min": base_cfg.p_min,
        },
        "files": {
            "figures_dir": figdir,
            "tables_dir": tabdir,
            "raw_dir": rawdir,
            "main_metric_table_manuscript_csv": os.path.join(tabdir, "main_metric_table_manuscript.csv"),
            "main_metric_table_manuscript_tex": os.path.join(tabdir, "main_metric_table_manuscript.tex"),
        },
        "n_main_scenarios": len(scenarios),
        "n_jam_sweep_points": len(jam_specs),
        "n_density_sweep_points": len(density_specs),
        "n_ablations": len(ablation_specs),
    }
    _save_json(os.path.join(outdir, "suite_summary.json"), summary)
    return summary

# ============================================================
# Demo main
# ============================================================




In [ ]:

# ============================================================
# Extended experiment helpers for the full manuscript campaign
# ============================================================

from pathlib import Path
from collections import defaultdict
import itertools
import warnings

import networkx as nx
from scipy.stats import wilcoxon

# -----------------------------
# Generic utilities
# -----------------------------

def stable_edge(edge):
    i, j = edge
    return (int(i), int(j)) if int(i) < int(j) else (int(j), int(i))

def adjacency_edge_list(A: np.ndarray) -> List[Tuple[int, int]]:
    n = A.shape[0]
    return [(i, j) for i in range(n) for j in range(i + 1, n) if A[i, j]]

def warning_score_from_result(res: ACLDResult, cfg: ACLDConfig) -> float:
    score_split = 1.0 if int(res.n_components_pred_viable) > 1 else 0.0
    score_lam = float(np.clip(
        (cfg.lambda2_partition_threshold - float(res.lambda2_pred_viable)) /
        max(cfg.lambda2_partition_threshold, 1e-9), 0.0, 1.0
    ))
    score_bridge = 1.0 if len(res.predicted_viable_bridges) > 0 else 0.0
    score_inter = float(np.clip(len(res.risky_inter) / 5.0, 0.0, 1.0))
    score = max(score_split, score_lam, 0.5 * score_bridge + 0.5 * score_inter)
    return float(np.clip(score, 0.0, 1.0))

def topk_recall_from_score_map(score_map: Dict[Tuple[int, int], float],
                               gt_edges: Iterable[Tuple[int, int]],
                               k: int) -> float:
    gt = {stable_edge(e) for e in gt_edges}
    if len(gt) == 0:
        return float("nan")
    ranked = sorted(score_map.items(), key=lambda kv: (-float(kv[1]), kv[0]))
    pred = {stable_edge(e) for e, _ in ranked[:max(1, int(k))]}
    hit = len(pred & gt)
    return float(hit / len(gt))

def bootstrap_mean_diff(a: np.ndarray, b: np.ndarray, n_boot: int = 5000,
                        seed: int = 42) -> Tuple[float, float, float]:
    rng = np.random.default_rng(seed)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    a = a[mask]
    b = b[mask]
    if len(a) == 0:
        return float("nan"), float("nan"), float("nan")
    diffs = a - b
    n = len(diffs)
    idx = rng.integers(0, n, size=(n_boot, n))
    boots = diffs[idx].mean(axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(np.mean(diffs)), float(lo), float(hi)

def paired_sign_effect(a: np.ndarray, b: np.ndarray) -> float:
    """Paired sign effect: (positive differences - negative differences) / N.

    This is not Cliff's delta; the explicit name prevents a misleading effect-size label.
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    d = a - b
    d = d[np.isfinite(d)]
    if len(d) == 0:
        return float("nan")
    pos = np.sum(d > 0)
    neg = np.sum(d < 0)
    return float((pos - neg) / len(d))

def cliffs_delta_paired(a: np.ndarray, b: np.ndarray) -> float:
    """Deprecated compatibility alias; use `paired_sign_effect`."""
    return paired_sign_effect(a, b)

def holm_adjust(p_values: Iterable[float]) -> np.ndarray:
    """Holm step-down family-wise error correction."""
    p = np.asarray(list(p_values), dtype=float)
    out = np.full_like(p, np.nan, dtype=float)
    finite = np.isfinite(p)
    idx = np.where(finite)[0]
    if len(idx) == 0:
        return out
    order = idx[np.argsort(p[idx])]
    m = len(order)
    running = 0.0
    for rank, original_idx in enumerate(order):
        adj = min(1.0, (m - rank) * float(p[original_idx]))
        running = max(running, adj)
        out[original_idx] = running
    return out

def reliability_bins(y_true: np.ndarray, y_score: np.ndarray, n_bins: int = 10) -> pd.DataFrame:
    y_true = np.asarray(y_true, dtype=float)
    y_score = np.asarray(y_score, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_score)
    y_true = y_true[mask]
    y_score = y_score[mask]
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    for b0, b1 in zip(bins[:-1], bins[1:]):
        if b1 < 1.0:
            m = (y_score >= b0) & (y_score < b1)
        else:
            m = (y_score >= b0) & (y_score <= b1)
        if not np.any(m):
            continue
        frac_pos = float(np.mean(y_true[m]))
        conf = float(np.mean(y_score[m]))
        rows.append({
            "bin_left": float(b0),
            "bin_right": float(b1),
            "count": int(np.sum(m)),
            "mean_score": conf,
            "empirical_rate": frac_pos,
            "abs_gap": abs(conf - frac_pos),
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    ece = float(np.average(df["abs_gap"], weights=df["count"]))
    df.attrs["ece"] = ece
    return df


def rician_kdb_to_nakagami_m(k_db: float) -> float:
    k_lin = 10.0 ** (float(k_db) / 10.0)
    return float(((k_lin + 1.0) ** 2) / (2.0 * k_lin + 1.0))

# -----------------------------
# Additional graph baselines
# -----------------------------

def betweenness_edge_scores(A: np.ndarray) -> Dict[Tuple[int, int], float]:
    G = nx.Graph()
    n = A.shape[0]
    G.add_nodes_from(range(n))
    for i, j in adjacency_edge_list(A):
        G.add_edge(i, j)
    if G.number_of_edges() == 0:
        return {}
    bc = nx.edge_betweenness_centrality(G, normalized=True)
    return {stable_edge(k): float(v) for k, v in bc.items()}

def cut_kec_edges_from_adjacency(A: np.ndarray) -> List[Tuple[int, int]]:
    G = nx.Graph()
    n = A.shape[0]
    G.add_nodes_from(range(n))
    G.add_edges_from(adjacency_edge_list(A))
    if G.number_of_edges() == 0:
        return []
    if not nx.is_connected(G):
        return [stable_edge(e) for e in ACLDCore.tarjan_bridges(A)]
    try:
        cut_value, partition = nx.stoer_wagner(G)
        S, T = partition
        edges = []
        for u in S:
            for v in T:
                if G.has_edge(u, v):
                    edges.append(stable_edge((u, v)))
        return sorted(set(edges))
    except Exception:
        return [stable_edge(e) for e in ACLDCore.tarjan_bridges(A)]

def cut_kec_edge_scores(A: np.ndarray) -> Dict[Tuple[int, int], float]:
    edges = cut_kec_edges_from_adjacency(A)
    return {stable_edge(e): 1.0 for e in edges}

# -----------------------------
# Method wrappers
# -----------------------------

MAIN_METHODS = [
    "acld_v3",
    "tarjan_snapshot_guard",
    "tarjan_predict_guard",
    "ebc_snapshot_guard",
    "cut_predict_guard",
]

ABLATION_METHODS = [
    "acld_v3",
    "acld_no_guard",
    "acld_no_forecast_bridge",
    "acld_no_adaptive_threshold",
]

def _prepare_shared_predictions(cfg: ACLDConfig, state: SwarmState):
    acld = ACLDCore(cfg)
    res = acld.run_acld(state)
    A_now_guard = build_current_viable_adjacency(acld, state, with_guard=True)
    A_now_geom = build_current_viable_adjacency(acld, state, with_guard=False)
    return acld, res, A_now_guard, A_now_geom

def _prepare_shared_predictions_timed(cfg: ACLDConfig, state: SwarmState):
    """Return shared context plus explicit timing scopes.

    `acld_pipeline_ms` times only `ACLDCore.run_acld`, matching the C2 scope.
    `shared_context_ms` includes the additional current-graph construction used
    by several baselines. These timings are kept separate from method-specific
    post-shared-preprocessing detector latency.
    """
    acld = ACLDCore(cfg)
    t0 = time.perf_counter()
    res = acld.run_acld(state)
    t1 = time.perf_counter()
    A_now_guard = build_current_viable_adjacency(acld, state, with_guard=True)
    A_now_geom = build_current_viable_adjacency(acld, state, with_guard=False)
    t2 = time.perf_counter()
    timing = {
        "acld_pipeline_ms": (t1 - t0) * 1000.0,
        "shared_context_ms": (t2 - t0) * 1000.0,
    }
    return (acld, res, A_now_guard, A_now_geom), timing

def detector_step_predictions(cfg: ACLDConfig,
                              state: SwarmState,
                              method: str,
                              rng: np.random.Generator,
                              shared=None) -> Dict[str, object]:
    if shared is None:
        acld, res, A_now_guard, A_now_geom = _prepare_shared_predictions(cfg, state)
    else:
        acld, res, A_now_guard, A_now_geom = shared

    if method == "acld_v3":
        edges = [stable_edge(e) for e in res.selected_critical_edges]
        score_map = {}
        for e in res.risky_intra:
            score_map[stable_edge(e)] = max(score_map.get(stable_edge(e), 0.0), 0.25)
        for e in res.risky_inter:
            score_map[stable_edge(e)] = max(score_map.get(stable_edge(e), 0.0), 0.75)
        for e in res.predicted_viable_bridges:
            score_map[stable_edge(e)] = max(score_map.get(stable_edge(e), 0.0), 1.00)
        alert = int(
            (res.n_components_pred_viable > 1)
            or (res.lambda2_pred_viable < cfg.lambda2_partition_threshold)
            or ((len(res.predicted_viable_bridges) > 0) and (len(res.risky_inter) > 0))
        )
        return {
            "alert": alert,
            "edges": edges,
            "edge_scores": score_map,
            "score": warning_score_from_result(res, cfg),
        }

    if method == "acld_no_guard":
        tmp_cfg = _clone_cfg(cfg, p_min=0.0)
        tmp_acld = ACLDCore(tmp_cfg)
        tmp_res = tmp_acld.run_acld(state)
        edges = [stable_edge(e) for e in tmp_res.selected_critical_edges]
        score_map = {stable_edge(e): 1.0 for e in edges}
        alert = int(
            (tmp_res.n_components_pred_viable > 1)
            or (tmp_res.lambda2_pred_viable < tmp_cfg.lambda2_partition_threshold)
            or ((len(tmp_res.predicted_viable_bridges) > 0) and (len(tmp_res.risky_inter) > 0))
        )
        return {"alert": alert, "edges": edges, "edge_scores": score_map,
                "score": warning_score_from_result(tmp_res, tmp_cfg)}

    if method == "acld_no_forecast_bridge":
        edges = sorted({stable_edge(e) for e in res.risky_inter})
        score_map = {stable_edge(e): 1.0 for e in edges}
        alert = int(
            (res.n_components_pred_viable > 1)
            or (res.lambda2_pred_viable < cfg.lambda2_partition_threshold)
            or (len(res.risky_inter) > 0)
        )
        return {"alert": alert, "edges": edges, "edge_scores": score_map,
                "score": warning_score_from_result(res, cfg)}

    if method == "acld_no_adaptive_threshold":
        tmp_cfg = _clone_cfg(cfg, alpha=0.0, beta=0.0, gamma=0.0, delta=0.0, zeta=0.0, risk_band_fraction=1.0)
        tmp_acld = ACLDCore(tmp_cfg)
        tmp_res = tmp_acld.run_acld(state)
        edges = [stable_edge(e) for e in tmp_res.selected_critical_edges]
        score_map = {stable_edge(e): 1.0 for e in edges}
        alert = int(
            (tmp_res.n_components_pred_viable > 1)
            or (tmp_res.lambda2_pred_viable < tmp_cfg.lambda2_partition_threshold)
            or ((len(tmp_res.predicted_viable_bridges) > 0) and (len(tmp_res.risky_inter) > 0))
        )
        return {"alert": alert, "edges": edges, "edge_scores": score_map,
                "score": warning_score_from_result(tmp_res, tmp_cfg)}

    if method == "tarjan_snapshot_guard":
        edges = [stable_edge(e) for e in ACLDCore.tarjan_bridges(A_now_guard)]
        return {"alert": int(len(edges) > 0), "edges": edges,
                "edge_scores": {stable_edge(e): 1.0 for e in edges},
                "score": float(min(1.0, len(edges) / 5.0))}

    if method == "tarjan_snapshot_geom":
        edges = [stable_edge(e) for e in ACLDCore.tarjan_bridges(A_now_geom)]
        return {"alert": int(len(edges) > 0), "edges": edges,
                "edge_scores": {stable_edge(e): 1.0 for e in edges},
                "score": float(min(1.0, len(edges) / 5.0))}

    if method == "tarjan_predict_guard":
        edges = [stable_edge(e) for e in res.predicted_viable_bridges]
        score_map = {stable_edge(e): 1.0 for e in edges}
        alert = int((res.n_components_pred_viable > 1) or (len(edges) > 0))
        return {"alert": alert, "edges": edges, "edge_scores": score_map,
                "score": float(max(1.0 if res.n_components_pred_viable > 1 else 0.0, min(1.0, len(edges) / 5.0)))}

    if method == "ebc_snapshot_guard":
        score_map = betweenness_edge_scores(A_now_guard)
        if len(score_map) == 0:
            return {"alert": 0, "edges": [], "edge_scores": {}, "score": 0.0}
        k = max(1, len(res.selected_critical_edges) if len(res.selected_critical_edges) > 0 else len(res.predicted_viable_bridges) or 1)
        ranked = sorted(score_map.items(), key=lambda kv: (-float(kv[1]), kv[0]))
        edges = [stable_edge(e) for e, _ in ranked[:k]]
        return {"alert": int(len(edges) > 0), "edges": edges,
                "edge_scores": score_map,
                "score": float(min(1.0, sum(score_map.values())))}

    if method == "cut_predict_guard":
        edges = [stable_edge(e) for e in cut_kec_edges_from_adjacency(res.A_pred_viable)]
        return {"alert": int((res.n_components_pred_viable > 1) or (len(edges) > 0)),
                "edges": edges,
                "edge_scores": {stable_edge(e): 1.0 for e in edges},
                "score": float(max(1.0 if res.n_components_pred_viable > 1 else 0.0, min(1.0, len(edges) / 5.0)))}

    if method == "random_k":
        all_edges = adjacency_edge_list(A_now_guard)
        if len(all_edges) == 0:
            return {"alert": 0, "edges": [], "edge_scores": {}, "score": 0.0}
        k = max(1, len(res.selected_critical_edges) if len(res.selected_critical_edges) > 0 else 1)
        idx = rng.choice(len(all_edges), size=min(k, len(all_edges)), replace=False)
        edges = [stable_edge(all_edges[ii]) for ii in np.atleast_1d(idx)]
        return {"alert": int(len(edges) > 0),
                "edges": edges,
                "edge_scores": {stable_edge(e): 1.0 for e in edges},
                "score": float(min(1.0, len(edges) / 5.0))}

    raise ValueError(f"Unknown method: {method}")

# -----------------------------
# Ground truth / boosted adjacency
# -----------------------------

def success_probability_uav_uav_with_boost(acld: ACLDCore, d_uav: np.ndarray,
                                           pos_for_jammer: Optional[np.ndarray],
                                           tx_boost_db: Optional[np.ndarray]) -> np.ndarray:
    cfg = acld.cfg
    d = np.maximum(d_uav, 1e-9)
    mean_snr = cfg.snr_0_lin * (cfg.d_0 / d) ** cfg.path_loss_exp
    if tx_boost_db is not None:
        boost = np.asarray(tx_boost_db[:cfg.n_uavs], dtype=float)
        bi = boost.reshape(-1, 1)
        bj = boost.reshape(1, -1)
        mean_snr = mean_snr * (10.0 ** ((bi + bj) / 10.0))
    jnr_lin_per_rx = None
    if pos_for_jammer is not None:
        jnr_lin_per_rx = acld._receiver_jnr_lin(pos_for_jammer)
    if jnr_lin_per_rx is not None:
        ji = jnr_lin_per_rx.reshape(-1, 1)
        jj = jnr_lin_per_rx.reshape(1, -1)
        j_lin_ij = np.maximum(ji, jj)
        mean_snr = mean_snr / (1.0 + j_lin_ij)
    s = acld._nakagami_success_from_mean_snr(mean_snr, cfg.m_fading)
    np.fill_diagonal(s, 0.0)
    return s

def success_probability_uav_gs_with_boost(acld: ACLDCore, d_gs: np.ndarray,
                                          pos: np.ndarray,
                                          tx_boost_db: Optional[np.ndarray]) -> np.ndarray:
    cfg = acld.cfg
    d = np.maximum(d_gs, 1e-9)
    mean_snr = cfg.gs_snr_0_lin * (cfg.d_0 / d) ** cfg.gs_path_loss_exp
    if tx_boost_db is not None:
        boost = np.asarray(tx_boost_db[:cfg.n_uavs], dtype=float)
        mean_snr = mean_snr * (10.0 ** (boost / 10.0))
    jnr_lin_per_rx = acld._receiver_jnr_lin(pos)
    if jnr_lin_per_rx is not None:
        mean_snr = mean_snr / (1.0 + jnr_lin_per_rx)
    s = acld._nakagami_success_from_mean_snr(mean_snr, cfg.gs_m_fading)
    return np.clip(s, 0.0, 1.0)

def build_current_viable_adjacency_with_boost(acld: ACLDCore, state: SwarmState,
                                              tx_boost_db: Optional[np.ndarray] = None,
                                              with_guard: bool = True) -> np.ndarray:
    cfg = acld.cfg
    n_uav = cfg.n_uavs
    n_total = cfg.n_total
    sidx = cfg.gs_index

    d_cur_uav = acld._pairwise_uav_dist(state.pos)
    _, _, theta_plus_uav = acld.adaptive_threshold_uav_uav(state, d_cur_uav)
    if with_guard:
        s_cur_uav = success_probability_uav_uav_with_boost(acld, d_cur_uav, state.pos, tx_boost_db)
    else:
        s_cur_uav = np.ones_like(d_cur_uav, dtype=float)

    A = np.zeros((n_total, n_total), dtype=np.uint8)
    mask_uav = (d_cur_uav <= theta_plus_uav)
    if with_guard:
        mask_uav &= (s_cur_uav >= cfg.p_min)
    np.fill_diagonal(mask_uav, False)
    A[:n_uav, :n_uav] = mask_uav.astype(np.uint8)

    if cfg.include_ground_station:
        d_cur_gs = acld._uav_gs_dist(state.pos)
        _, _, theta_plus_gs = acld.adaptive_threshold_uav_gs(state, d_cur_uav)
        if with_guard:
            s_cur_gs = success_probability_uav_gs_with_boost(acld, d_cur_gs, state.pos, tx_boost_db)
        else:
            s_cur_gs = np.ones_like(d_cur_gs, dtype=float)

        for i in range(n_uav):
            ok = bool(d_cur_gs[i] <= theta_plus_gs[i])
            if with_guard:
                ok = ok and bool(s_cur_gs[i] >= cfg.p_min)
            if ok:
                A[i, sidx] = 1
                A[sidx, i] = 1
    return A

def edge_f1_against_next_graph(A_next_viable: np.ndarray,
                               predicted_edges: List[Tuple[int, int]],
                               rng: Optional[np.random.Generator] = None,
                               exact_n_threshold: Optional[int] = None,
                               sample_size: Optional[int] = None) -> Tuple[float, List[Tuple[int, int]]]:
    """
    Exact manuscript Edge-F1.

    Ground truth is the complete Tarjan bridge set of the realized next-step
    channel-viable graph for every swarm size. The former sampled approximation
    is intentionally not used because unsampled flagged edges could otherwise be
    counted as false positives. The legacy threshold/sample arguments are kept
    only for call compatibility and are ignored.
    """
    metrics = _edge_f1_exact_vs_bridges(A_next_viable, predicted_edges)
    gt = [stable_edge(e) for e in ACLDCore.tarjan_bridges(A_next_viable)]
    return float(metrics["f1"]), gt

# -----------------------------
# Detection / localization campaign
# -----------------------------

@dataclass
class CampaignSpec:
    name: str
    cfg_overrides: Dict[str, object]
    horizon_s: float
    seeds: Tuple[int, ...]
    methods: Tuple[str, ...] = tuple(MAIN_METHODS)

def run_detection_episode(cfg: ACLDConfig,
                          methods: Tuple[str, ...],
                          horizon_s: float,
                          seed: int,
                          store_step_rows: bool = True) -> Tuple[List[Dict[str, object]], List[Dict[str, object]]]:
    rng = make_rng(seed)
    state = SwarmState.random_init(cfg, rng)
    T = int(round(horizon_s / cfg.dt_sim))

    y_true_map = {m: [] for m in methods}
    yhat_map = {m: [] for m in methods}
    score_map = {m: [] for m in methods}
    edge_f1_map = {m: [] for m in methods}  # diagnostic; NaN when no localization support
    edge_tp_map = {m: [] for m in methods}
    edge_fp_map = {m: [] for m in methods}
    edge_fn_map = {m: [] for m in methods}
    cand_map = {m: [] for m in methods}
    top1_map = {m: [] for m in methods}
    top3_map = {m: [] for m in methods}
    top5_map = {m: [] for m in methods}
    incremental_runtime_map = {m: [] for m in methods}
    acld_pipeline_runtime = []
    shared_context_runtime = []

    step_rows = []

    for step_idx in range(max(0, T - 1)):
        shared, shared_timing = _prepare_shared_predictions_timed(cfg, state)
        acld_shared, res, A_now_guard, A_now_geom = shared
        acld_pipeline_runtime.append(float(shared_timing["acld_pipeline_ms"]))
        shared_context_runtime.append(float(shared_timing["shared_context_ms"]))

        predictions = {}
        for method in methods:
            t0 = time.perf_counter()
            pred = detector_step_predictions(cfg, state, method, rng, shared=shared)
            inc_ms = (time.perf_counter() - t0) * 1000.0
            incremental_runtime_map[method].append(inc_ms)
            predictions[method] = pred

        next_state = state.copy()
        next_state.step_random_kinematics(cfg, rng)
        A_next_viable = build_current_viable_adjacency(acld_shared, next_state, with_guard=True)
        n_comp_next, _ = connectivity_components_from_adjacency(A_next_viable)
        y_true = int(n_comp_next > 1)

        # Compute exact ground-truth bridges once per step and reuse for all methods.
        gt_bridges = [stable_edge(e) for e in ACLDCore.tarjan_bridges(A_next_viable)]
        gt_bridge_set = set(gt_bridges)

        for method in methods:
            pred = predictions[method]
            metrics = _edge_f1_exact_vs_bridges(A_next_viable, pred["edges"])
            f1 = float(metrics["f1"])

            y_true_map[method].append(y_true)
            yhat_map[method].append(int(pred["alert"]))
            score_map[method].append(float(pred.get("score", np.nan)))
            edge_f1_map[method].append(f1)
            edge_tp_map[method].append(int(metrics["tp"]))
            edge_fp_map[method].append(int(metrics["fp"]))
            edge_fn_map[method].append(int(metrics["fn"]))
            cand_map[method].append(int(len(pred["edges"])))
            top1_map[method].append(topk_recall_from_score_map(pred.get("edge_scores", {}), gt_bridges, 1))
            top3_map[method].append(topk_recall_from_score_map(pred.get("edge_scores", {}), gt_bridges, 3))
            top5_map[method].append(topk_recall_from_score_map(pred.get("edge_scores", {}), gt_bridges, 5))

            if store_step_rows:
                step_rows.append({
                    "step_idx": int(step_idx),
                    "t": float(step_idx * cfg.dt_sim),
                    "method": method,
                    "label_partition_next": int(y_true),
                    "alert": int(pred["alert"]),
                    "warning_score": float(pred.get("score", np.nan)),
                    "edge_f1_step": f1,
                    "candidate_edges": int(len(pred["edges"])),
                    "gt_bridge_count": int(len(gt_bridge_set)),
                    "top1_recall": top1_map[method][-1],
                    "top3_recall": top3_map[method][-1],
                    "top5_recall": top5_map[method][-1],
                    # Diagnostic latency only. Cross-method speed ranking is invalid because
                    # methods reuse different portions of the shared ACLD/prediction context.
                    "runtime_incremental_ms": float(incremental_runtime_map[method][-1]),
                    "runtime_ms": float(incremental_runtime_map[method][-1]),  # legacy diagnostic alias
                    # Full ACLD core timing applies only to the ACLD-v3 row.  It is not
                    # copied to rivals/ablations as if it were their end-to-end runtime.
                    "acld_pipeline_ms": (
                        float(shared_timing["acld_pipeline_ms"])
                        if method == "acld_v3" else float("nan")
                    ),
                    # Common context timing is repeated only as provenance/context, not
                    # as method latency.
                    "shared_context_ms": float(shared_timing["shared_context_ms"]),
                    "runtime_scope": "diagnostic_post_shared_context_not_cross_method_comparable",
                    "runtime_comparable_across_methods": False,
                    "lambda2_current": float(res.lambda2_current),
                    "lambda2_pred_viable": float(res.lambda2_pred_viable),
                    "n_components_pred_viable": int(res.n_components_pred_viable),
                    "n_risky_inter": int(len(res.risky_inter)),
                    "n_risky_intra": int(len(res.risky_intra)),
                })

        state = next_state

    episode_rows = []
    acld_pipe_med = float(np.nanmedian(acld_pipeline_runtime)) if acld_pipeline_runtime else float("nan")
    acld_pipe_p95 = float(np.nanpercentile(acld_pipeline_runtime, 95)) if acld_pipeline_runtime else float("nan")
    shared_med = float(np.nanmedian(shared_context_runtime)) if shared_context_runtime else float("nan")
    shared_p95 = float(np.nanpercentile(shared_context_runtime, 95)) if shared_context_runtime else float("nan")

    def _safe_finite_mean(values):
        arr = np.asarray(values, dtype=float)
        arr = arr[np.isfinite(arr)]
        return float(np.mean(arr)) if arr.size else float("nan")

    for method in methods:
        conf = _binary_confusion_metrics(y_true_map[method], yhat_map[method])
        inc_med = float(np.nanmedian(incremental_runtime_map[method])) if incremental_runtime_map[method] else float("nan")
        inc_p95 = float(np.nanpercentile(incremental_runtime_map[method], 95)) if incremental_runtime_map[method] else float("nan")

        tp_total = int(np.sum(edge_tp_map[method], dtype=np.int64)) if edge_tp_map[method] else 0
        fp_total = int(np.sum(edge_fp_map[method], dtype=np.int64)) if edge_fp_map[method] else 0
        fn_total = int(np.sum(edge_fn_map[method], dtype=np.int64)) if edge_fn_map[method] else 0
        edge_micro = _prf_from_edge_counts(tp_total, fp_total, fn_total)
        edge_support_steps = int(sum(
            (tp + fp + fn) > 0
            for tp, fp, fn in zip(edge_tp_map[method], edge_fp_map[method], edge_fn_map[method])
        ))

        episode_rows.append({
            "seed": int(seed),
            "method": method,
            "pew_precision": float(conf["precision"]),
            "pew_recall": float(conf["recall"]),
            "pew_f1": float(conf["f1"]),
            "fpr": float(conf["fpr"]),
            "acc": float(conf["acc"]),
            "partition_next_rate": float(np.mean(y_true_map[method])) if y_true_map[method] else float("nan"),
            "warning_score_mean": _safe_finite_mean(score_map[method]),
            # Primary exact localization metric: micro-F1 from exact TP/FP/FN
            # accumulated across the episode.  `edge_f1_mean` is retained as a
            # compatibility alias for downstream table builders.
            "edge_precision_micro": float(edge_micro["precision"]),
            "edge_recall_micro": float(edge_micro["recall"]),
            "edge_f1_micro": float(edge_micro["f1"]),
            "edge_f1_mean": float(edge_micro["f1"]),
            "edge_tp_total": tp_total,
            "edge_fp_total": fp_total,
            "edge_fn_total": fn_total,
            "edge_support_steps": edge_support_steps,
            "edge_f1_step_supported_mean": _safe_finite_mean(edge_f1_map[method]),
            "candidate_edges_mean": _safe_finite_mean(cand_map[method]),
            "top1_recall_mean": _safe_finite_mean(top1_map[method]),
            "top3_recall_mean": _safe_finite_mean(top3_map[method]),
            "top5_recall_mean": _safe_finite_mean(top5_map[method]),
            "runtime_incremental_median_ms": inc_med,
            "runtime_incremental_p95_ms": inc_p95,
            "runtime_median_ms": inc_med,  # legacy diagnostic alias; never rank methods with it
            "runtime_p95_ms": inc_p95,
            "acld_pipeline_median_ms": acld_pipe_med if method == "acld_v3" else float("nan"),
            "acld_pipeline_p95_ms": acld_pipe_p95 if method == "acld_v3" else float("nan"),
            "shared_context_median_ms": shared_med,
            "shared_context_p95_ms": shared_p95,
            "runtime_scope": "diagnostic_post_shared_context_not_cross_method_comparable",
            "runtime_comparable_across_methods": False,
        })
    return episode_rows, step_rows

def run_detection_campaign(base_cfg: ACLDConfig,
                           specs: List[CampaignSpec],
                           outdir: str,
                           overwrite: bool = False,
                           store_step_rows: bool = True) -> Dict[str, pd.DataFrame]:
    outdir = str(outdir)
    Path(outdir).mkdir(parents=True, exist_ok=True)
    episode_rows_all = []
    step_rows_all = []

    for spec in specs:
        print(f"[A] running scenario: {spec.name}")
        cfg = _clone_cfg(base_cfg, **spec.cfg_overrides)
        for seed in spec.seeds:
            ep_rows, st_rows = run_detection_episode(cfg, spec.methods, spec.horizon_s, seed, store_step_rows=store_step_rows)
            for r in ep_rows:
                rr = dict(r)
                rr["scenario"] = spec.name
                rr["n_uavs"] = int(cfg.n_uavs)
                rr["density_scale_mode"] = str(cfg.density_scale_mode)
                rr["jnr_db"] = cfg.jnr_db if cfg.jnr_db is not None else "NoJam"
                rr["alpha"] = float(cfg.alpha)
                rr["risk_band_fraction"] = float(cfg.risk_band_fraction)
                episode_rows_all.append(rr)
            for r in st_rows:
                rr = dict(r)
                rr["scenario"] = spec.name
                rr["seed"] = int(seed)
                rr["n_uavs"] = int(cfg.n_uavs)
                rr["density_scale_mode"] = str(cfg.density_scale_mode)
                rr["jnr_db"] = cfg.jnr_db if cfg.jnr_db is not None else "NoJam"
                rr["alpha"] = float(cfg.alpha)
                rr["risk_band_fraction"] = float(cfg.risk_band_fraction)
                step_rows_all.append(rr)

    ep_df = pd.DataFrame(episode_rows_all)
    st_df = pd.DataFrame(step_rows_all)

    agg = (ep_df.groupby(["scenario", "method", "n_uavs", "density_scale_mode", "jnr_db"], dropna=False)
             .agg(
                 n_episodes=("seed", "nunique"),
                 pew_f1_mean=("pew_f1", "mean"),
                 pew_f1_std=("pew_f1", "std"),
                 fpr_mean=("fpr", "mean"),
                 fpr_std=("fpr", "std"),
                 edge_f1_mean=("edge_f1_mean", "mean"),
                 edge_f1_std=("edge_f1_mean", "std"),
                 cand_mean=("candidate_edges_mean", "mean"),
                 cand_std=("candidate_edges_mean", "std"),
                 runtime_median_ms_mean=("runtime_median_ms", "mean"),
                 runtime_median_ms_std=("runtime_median_ms", "std"),
                 top1_recall_mean=("top1_recall_mean", "mean"),
                 top3_recall_mean=("top3_recall_mean", "mean"),
                 top5_recall_mean=("top5_recall_mean", "mean"),
                 prevalence_mean=("partition_next_rate", "mean"),
             )
             .reset_index())

    ep_df.to_csv(Path(outdir) / "detection_episode_metrics_long.csv", index=False)
    if not st_df.empty:
        st_df.to_csv(Path(outdir) / "detection_step_metrics_long.csv", index=False)
    agg.to_csv(Path(outdir) / "detection_summary_long.csv", index=False)
    return {"episode": ep_df, "step": st_df, "summary": agg}

# -----------------------------
# Closed-loop mitigation campaign
# -----------------------------

def pick_top_edges(edge_scores: Dict[Tuple[int, int], float], edge_budget: int) -> List[Tuple[int, int]]:
    if len(edge_scores) == 0:
        return []
    ranked = sorted(edge_scores.items(), key=lambda kv: (-float(kv[1]), kv[0]))
    return [stable_edge(e) for e, _ in ranked[:max(1, int(edge_budget))]]

def apply_micro_trajectory_tweak(state: SwarmState,
                                 edges: List[Tuple[int, int]],
                                 cfg: ACLDConfig,
                                 dv_step: float = 1.0,
                                 max_nodes_per_tick: int = 4) -> Tuple[int, float]:
    """
    Apply a lightweight velocity nudge on actionable endpoints.

    Important:
    - SwarmState stores UAV states only (size = cfg.n_uavs).
    - Some critical edges may include the ground station, whose index is cfg.gs_index (= cfg.n_uavs).
    - For UAV-GS edges, use the fixed ground-station position as the geometric anchor and update
      only the UAV endpoint. This avoids indexing state.pos/state.vel with the GS node.
    """
    touched = set()
    total_dv = 0.0
    gs_idx = cfg.gs_index

    def _endpoint_pos(idx: int) -> Optional[np.ndarray]:
        idx = int(idx)
        if 0 <= idx < cfg.n_uavs:
            return state.pos[idx]
        if cfg.include_ground_station and gs_idx is not None and idx == gs_idx:
            return np.asarray(cfg.ground_station_pos, dtype=float)
        return None

    for i, j in edges:
        if len(touched) >= max_nodes_per_tick:
            break

        i = int(i)
        j = int(j)

        pi = _endpoint_pos(i)
        pj = _endpoint_pos(j)
        if pi is None or pj is None:
            continue

        delta = pj - pi
        norm = float(np.linalg.norm(delta))
        if norm < 1e-9:
            continue
        u = delta / norm

        if (0 <= i < cfg.n_uavs) and (i not in touched) and len(touched) < max_nodes_per_tick:
            state.vel[i] = state.vel[i] + dv_step * u
            touched.add(i)
            total_dv += dv_step

        if (0 <= j < cfg.n_uavs) and (j not in touched) and len(touched) < max_nodes_per_tick:
            state.vel[j] = state.vel[j] - dv_step * u
            touched.add(j)
            total_dv += dv_step

    state.vel = _clip_rows_norm(state.vel, cfg.v_max)
    return len(touched), total_dv

def allocate_tx_boost(cfg: ACLDConfig,
                      edges: List[Tuple[int, int]],
                      edge_boost_db: float = 3.0,
                      max_nodes_per_tick: int = 4) -> Tuple[np.ndarray, float]:
    boost = np.zeros(cfg.n_total, dtype=float)
    touched = set()
    total_boost = 0.0
    for i, j in edges:
        for node in (i, j):
            if node < cfg.n_uavs and (node not in touched) and len(touched) < max_nodes_per_tick:
                boost[node] += edge_boost_db / 2.0
                touched.add(node)
                total_boost += edge_boost_db / 2.0
    return boost, total_boost

def select_controller_edges(cfg: ACLDConfig,
                            state: SwarmState,
                            controller: str,
                            rng: np.random.Generator,
                            edge_budget: int = 2) -> Tuple[List[Tuple[int, int]], Dict[str, object]]:
    shared = _prepare_shared_predictions(cfg, state)
    acld, res, A_now_guard, _ = shared
    if controller == "nh":
        return [], {"source": "none"}
    if controller == "random":
        all_edges = adjacency_edge_list(A_now_guard)
        if len(all_edges) == 0:
            return [], {"source": "random"}
        idx = rng.choice(len(all_edges), size=min(edge_budget, len(all_edges)), replace=False)
        return [stable_edge(all_edges[ii]) for ii in np.atleast_1d(idx)], {"source": "random"}
    method = {
        "acld": "acld_v3",
        "tarjan_snapshot": "tarjan_snapshot_guard",
        "tarjan_predict": "tarjan_predict_guard",
        "ebc": "ebc_snapshot_guard",
        "cut": "cut_predict_guard",
    }.get(controller, controller)
    pred = detector_step_predictions(cfg, state, method, rng, shared=shared)
    edges = pick_top_edges(pred.get("edge_scores", {}), edge_budget)
    if len(edges) == 0:
        edges = pred.get("edges", [])[:edge_budget]
    return [stable_edge(e) for e in edges], pred

def run_closed_loop_episode(cfg: ACLDConfig,
                            controller: str,
                            horizon_s: float,
                            seed: int,
                            intervention_mode: str = "hybrid",
                            edge_budget: int = 2,
                            max_nodes_per_tick: int = 4,
                            dv_step: float = 1.0,
                            edge_boost_db: float = 3.0) -> Tuple[List[Dict[str, object]], Dict[str, object]]:
    rng = make_rng(seed)
    acld = ACLDCore(cfg)
    state = SwarmState.random_init(cfg, rng)
    T = int(round(horizon_s / cfg.dt_sim))
    step_rows = []
    partitions = 0
    connected_ticks = 0
    lambda2_vals = []
    total_nodes_touched = 0
    total_dv = 0.0
    total_boost = 0.0
    total_actions = 0

    for step_idx in range(max(0, T - 1)):
        chosen_edges, meta = select_controller_edges(cfg, state, controller, rng, edge_budget=edge_budget)
        next_state = state.copy()
        nodes_touched = 0
        dv_used = 0.0
        tx_boost = np.zeros(cfg.n_total, dtype=float)
        boost_used = 0.0

        if intervention_mode in ("micro", "hybrid") and len(chosen_edges) > 0:
            nodes_touched, dv_used = apply_micro_trajectory_tweak(
                next_state, chosen_edges, cfg, dv_step=dv_step, max_nodes_per_tick=max_nodes_per_tick
            )
        if intervention_mode in ("power", "hybrid") and len(chosen_edges) > 0:
            tx_boost, boost_used = allocate_tx_boost(
                cfg, chosen_edges, edge_boost_db=edge_boost_db, max_nodes_per_tick=max_nodes_per_tick
            )

        next_state.step_random_kinematics(cfg, rng)
        A_next = build_current_viable_adjacency_with_boost(acld, next_state, tx_boost_db=tx_boost, with_guard=True)
        n_comp_next, _ = connectivity_components_from_adjacency(A_next)
        lam_next = binary_lambda2_from_adjacency(A_next)

        disconnected = int(n_comp_next > 1)
        partitions += disconnected
        connected_ticks += int(not disconnected)
        lambda2_vals.append(float(lam_next))

        total_nodes_touched += int(nodes_touched)
        total_dv += float(dv_used)
        total_boost += float(boost_used)
        total_actions += int(len(chosen_edges) > 0)

        step_rows.append({
            "step_idx": int(step_idx),
            "t": float(step_idx * cfg.dt_sim),
            "controller": controller,
            "intervention_mode": intervention_mode,
            "chosen_edges": int(len(chosen_edges)),
            "nodes_touched": int(nodes_touched),
            "dv_used": float(dv_used),
            "boost_db_used": float(boost_used),
            "partition_next": int(disconnected),
            "lambda2_next_bin": float(lam_next),
            "n_components_next": int(n_comp_next),
        })
        state = next_state

    summary = {
        "seed": int(seed),
        "controller": controller,
        "intervention_mode": intervention_mode,
        "partitions": int(partitions),
        "connected_ratio": float(connected_ticks / max(1, T - 1)),
        "mean_lambda2_bin": float(np.mean(lambda2_vals)) if lambda2_vals else float("nan"),
        "actions_per_episode": int(total_actions),
        "nodes_touched_total": int(total_nodes_touched),
        "dv_total": float(total_dv),
        "boost_db_total": float(total_boost),
    }
    return step_rows, summary

def run_closed_loop_campaign(base_cfg: ACLDConfig,
                             specs: List[CampaignSpec],
                             controllers: Tuple[str, ...],
                             outdir: str,
                             intervention_mode: str = "hybrid") -> Dict[str, pd.DataFrame]:
    Path(outdir).mkdir(parents=True, exist_ok=True)
    step_rows_all = []
    episode_rows_all = []
    for spec in specs:
        print(f"[B] running mitigation scenario: {spec.name}")
        cfg = _clone_cfg(base_cfg, **spec.cfg_overrides)
        for controller in controllers:
            for seed in spec.seeds:
                st_rows, summ = run_closed_loop_episode(cfg, controller, spec.horizon_s, seed, intervention_mode=intervention_mode)
                for r in st_rows:
                    rr = dict(r)
                    rr["scenario"] = spec.name
                    rr["seed"] = int(seed)
                    rr["n_uavs"] = int(cfg.n_uavs)
                    rr["density_scale_mode"] = str(cfg.density_scale_mode)
                    rr["jnr_db"] = cfg.jnr_db if cfg.jnr_db is not None else "NoJam"
                    step_rows_all.append(rr)
                rr = dict(summ)
                rr["scenario"] = spec.name
                rr["n_uavs"] = int(cfg.n_uavs)
                rr["density_scale_mode"] = str(cfg.density_scale_mode)
                rr["jnr_db"] = cfg.jnr_db if cfg.jnr_db is not None else "NoJam"
                episode_rows_all.append(rr)

    ep_df = pd.DataFrame(episode_rows_all)
    st_df = pd.DataFrame(step_rows_all)
    agg = (ep_df.groupby(["scenario", "controller", "intervention_mode", "n_uavs", "density_scale_mode", "jnr_db"], dropna=False)
             .agg(
                 n_episodes=("seed", "nunique"),
                 partitions_mean=("partitions", "mean"),
                 partitions_std=("partitions", "std"),
                 connected_ratio_mean=("connected_ratio", "mean"),
                 connected_ratio_std=("connected_ratio", "std"),
                 mean_lambda2_bin_mean=("mean_lambda2_bin", "mean"),
                 mean_lambda2_bin_std=("mean_lambda2_bin", "std"),
                 actions_per_episode_mean=("actions_per_episode", "mean"),
                 nodes_touched_total_mean=("nodes_touched_total", "mean"),
                 dv_total_mean=("dv_total", "mean"),
                 boost_db_total_mean=("boost_db_total", "mean"),
             )
             .reset_index())
    ep_df.to_csv(Path(outdir) / "mitigation_episode_metrics_long.csv", index=False)
    st_df.to_csv(Path(outdir) / "mitigation_step_metrics_long.csv", index=False)
    agg.to_csv(Path(outdir) / "mitigation_summary_long.csv", index=False)
    return {"episode": ep_df, "step": st_df, "summary": agg}

# -----------------------------
# Significance / prevalence / calibration / failure analysis
# -----------------------------

def run_significance_suite(ep_df: pd.DataFrame,
                           ref_method: str = "acld_v3",
                           metrics: Tuple[str, ...] = ("pew_f1", "edge_f1_mean", "fpr"),
                           outpath: Optional[str] = None) -> pd.DataFrame:
    rows = []
    for scenario in sorted(ep_df["scenario"].dropna().unique()):
        sub = ep_df[ep_df["scenario"] == scenario].copy()
        methods = [m for m in sorted(sub["method"].dropna().unique()) if m != ref_method]
        ref = sub[sub["method"] == ref_method].set_index("seed")
        for method in methods:
            cmp = sub[sub["method"] == method].set_index("seed")
            common = sorted(set(ref.index) & set(cmp.index))
            if not common:
                continue
            for metric in metrics:
                a = ref.loc[common, metric].to_numpy(dtype=float)
                b = cmp.loc[common, metric].to_numpy(dtype=float)
                mean_diff, ci_lo, ci_hi = bootstrap_mean_diff(a, b)
                try:
                    stat, p = wilcoxon(a, b, zero_method="wilcox", alternative="two-sided")
                except Exception:
                    stat, p = np.nan, np.nan
                rows.append({
                    "scenario": scenario,
                    "metric": metric,
                    "reference": ref_method,
                    "comparator": method,
                    "n_pairs": len(common),
                    "mean_diff": mean_diff,
                    "ci_lo": ci_lo,
                    "ci_hi": ci_hi,
                    "wilcoxon_stat": float(stat) if stat == stat else np.nan,
                    "wilcoxon_p": float(p) if p == p else np.nan,
                    "cliffs_delta_paired": cliffs_delta_paired(a, b),
                })
    df = pd.DataFrame(rows)
    if outpath is not None:
        df.to_csv(outpath, index=False)
    return df

def run_scenario_paired_significance_suite(ep_df: pd.DataFrame,
                                           ref_method: str = "acld_v3",
                                           metrics: Tuple[str, ...] = ("pew_f1", "edge_f1_mean", "fpr"),
                                           outpath: Optional[str] = None) -> pd.DataFrame:
    """Scenario-balanced paired inference.

    Each scenario contributes one paired observation per method/metric after
    averaging its available paired seeds. This aligns inference with the
    scenario-wise common-frontier comparison and avoids treating thousands of
    within-scenario time steps as independent evidence. Holm correction is
    applied across the resulting comparator-by-metric family.
    """
    rows = []
    work = ep_df.copy()
    for metric in metrics:
        agg = (work.groupby(["scenario", "method"], as_index=False)[metric]
                    .mean())
        ref = agg[agg["method"] == ref_method].set_index("scenario")
        comparators = [m for m in sorted(agg["method"].dropna().unique()) if m != ref_method]
        for method in comparators:
            cmp = agg[agg["method"] == method].set_index("scenario")
            common = sorted(set(ref.index) & set(cmp.index))
            if not common:
                continue
            a = ref.loc[common, metric].to_numpy(dtype=float)
            b = cmp.loc[common, metric].to_numpy(dtype=float)
            # For FPR, lower is better. Keep raw ACLD-comparator difference for transparency.
            mean_diff, ci_lo, ci_hi = bootstrap_mean_diff(a, b)
            try:
                stat, p = wilcoxon(a, b, zero_method="wilcox", alternative="two-sided")
            except Exception:
                stat, p = np.nan, np.nan
            rows.append({
                "metric": metric,
                "reference": ref_method,
                "comparator": method,
                "n_scenarios": len(common),
                "mean_diff_ref_minus_cmp": mean_diff,
                "ci_lo": ci_lo,
                "ci_hi": ci_hi,
                "wilcoxon_stat": float(stat) if stat == stat else np.nan,
                "wilcoxon_p": float(p) if p == p else np.nan,
                "paired_sign_effect": paired_sign_effect(a, b),
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        df["wilcoxon_p_holm"] = holm_adjust(df["wilcoxon_p"].to_numpy())
    if outpath is not None:
        df.to_csv(outpath, index=False)
    return df

def prevalence_control_analysis(step_df: pd.DataFrame,
                                method: str = "acld_v3",
                                group_col: str = "scenario",
                                n_repeats: int = 100,
                                seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    sub = step_df[step_df["method"] == method].copy()
    rows = []
    for group_val, g in sub.groupby(group_col):
        pos = g[g["label_partition_next"] == 1]
        neg = g[g["label_partition_next"] == 0]
        if len(pos) == 0 or len(neg) == 0:
            continue
        m = min(len(pos), len(neg))
        for rep in range(n_repeats):
            pos_s = pos.sample(n=m, replace=False, random_state=int(rng.integers(0, 1_000_000)))
            neg_s = neg.sample(n=m, replace=False, random_state=int(rng.integers(0, 1_000_000)))
            gg = pd.concat([pos_s, neg_s], axis=0)
            conf = _binary_confusion_metrics(gg["label_partition_next"].tolist(), gg["alert"].tolist())
            rows.append({
                group_col: group_val,
                "repeat": int(rep),
                "balanced_f1": float(conf["f1"]),
                "balanced_precision": float(conf["precision"]),
                "balanced_recall": float(conf["recall"]),
                "balanced_fpr": float(conf["fpr"]),
                "prevalence_original": float(g["label_partition_next"].mean()),
                "n_each_class": int(m),
            })
    return pd.DataFrame(rows)

def calibration_analysis(step_df: pd.DataFrame,
                         method: str = "acld_v3",
                         outdir: Optional[str] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    sub = step_df[step_df["method"] == method].copy()
    overall = reliability_bins(sub["label_partition_next"].to_numpy(), sub["warning_score"].to_numpy(), n_bins=10)
    by_scenario_rows = []
    for scenario, g in sub.groupby("scenario"):
        bins = reliability_bins(g["label_partition_next"].to_numpy(), g["warning_score"].to_numpy(), n_bins=10)
        ece = bins.attrs.get("ece", np.nan) if not bins.empty else np.nan
        by_scenario_rows.append({"scenario": scenario, "ece": ece, "n_rows": len(g)})
    by_scenario = pd.DataFrame(by_scenario_rows)
    if outdir is not None:
        if not overall.empty:
            overall.to_csv(Path(outdir) / "calibration_overall_bins.csv", index=False)
        by_scenario.to_csv(Path(outdir) / "calibration_by_scenario.csv", index=False)
    return overall, by_scenario

def failure_mode_analysis(step_df: pd.DataFrame,
                          method: str = "acld_v3",
                          high_cand_quantile: float = 0.95,
                          outpath: Optional[str] = None) -> pd.DataFrame:
    sub = step_df[step_df["method"] == method].copy()
    if sub.empty:
        return pd.DataFrame()
    q = sub["candidate_edges"].quantile(high_cand_quantile)
    rows = []
    for _, r in sub.iterrows():
        if int(r["label_partition_next"]) == 1 and int(r["alert"]) == 0:
            failure = "false_negative"
        elif int(r["label_partition_next"]) == 0 and int(r["alert"]) == 1:
            failure = "false_positive"
        elif float(r["candidate_edges"]) >= q:
            failure = "candidate_explosion"
        elif float(r["lambda2_pred_viable"]) < 1e-3 and int(r["label_partition_next"]) == 0:
            failure = "lambda2_over_warning"
        else:
            failure = "ok"
        rows.append({
            "scenario": r["scenario"],
            "seed": int(r["seed"]),
            "step_idx": int(r["step_idx"]),
            "failure_mode": failure,
            "candidate_edges": int(r["candidate_edges"]),
            "gt_bridge_count": int(r["gt_bridge_count"]),
            "lambda2_pred_viable": float(r["lambda2_pred_viable"]),
            "n_components_pred_viable": int(r["n_components_pred_viable"]),
        })
    df = pd.DataFrame(rows)
    summary = (df.groupby(["scenario", "failure_mode"])
                 .size()
                 .reset_index(name="count")
                 .sort_values(["scenario", "count"], ascending=[True, False]))
    if outpath is not None:
        summary.to_csv(outpath, index=False)
    return summary

# -----------------------------
# Plot helpers for the new campaign
# -----------------------------

def plot_detection_summary(summary_df: pd.DataFrame, outdir: str) -> None:
    Path(outdir).mkdir(parents=True, exist_ok=True)
    metrics = [("pew_f1_mean", "PEW-F1"), ("fpr_mean", "FPR"), ("edge_f1_mean", "Edge-F1")]
    for metric, label in metrics:
        plt.figure(figsize=(12, 5))
        order = list(summary_df["scenario"].drop_duplicates())
        methods = list(summary_df["method"].drop_duplicates())
        width = 0.15
        x = np.arange(len(order))
        for idx, method in enumerate(methods):
            sub = summary_df[summary_df["method"] == method].set_index("scenario")
            vals = [sub.loc[s, metric] if s in sub.index else np.nan for s in order]
            plt.bar(x + idx * width, vals, width=width, label=method)
        plt.xticks(x + width * (len(methods) - 1) / 2.0, order, rotation=25, ha="right")
        plt.ylabel(label)
        plt.title(f"{label} across scenarios")
        plt.legend()
        plt.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig(Path(outdir) / f"detection_{metric}.png", dpi=180)
        plt.close()

def plot_mitigation_summary(summary_df: pd.DataFrame, outdir: str) -> None:
    Path(outdir).mkdir(parents=True, exist_ok=True)
    for metric, label in [("partitions_mean", "Partition incidents"), ("connected_ratio_mean", "Connected ratio")]:
        plt.figure(figsize=(10, 5))
        order = list(summary_df["scenario"].drop_duplicates())
        controllers = list(summary_df["controller"].drop_duplicates())
        width = 0.16
        x = np.arange(len(order))
        for idx, ctrl in enumerate(controllers):
            sub = summary_df[summary_df["controller"] == ctrl].set_index("scenario")
            vals = [sub.loc[s, metric] if s in sub.index else np.nan for s in order]
            plt.bar(x + idx * width, vals, width=width, label=ctrl)
        plt.xticks(x + width * (len(controllers) - 1) / 2.0, order, rotation=25, ha="right")
        plt.ylabel(label)
        plt.title(f"{label} by controller")
        plt.legend()
        plt.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig(Path(outdir) / f"mitigation_{metric}.png", dpi=180)
        plt.close()

def plot_scaling_accuracy_runtime(ep_df: pd.DataFrame, outdir: str) -> None:
    Path(outdir).mkdir(parents=True, exist_ok=True)
    sub = ep_df.copy()
    for metric, label in [("pew_f1", "PEW-F1"), ("edge_f1_mean", "Edge-F1"), ("runtime_median_ms", "Runtime median (ms)")]:
        plt.figure(figsize=(10, 5))
        for method, g in sub.groupby("method"):
            agg = g.groupby("n_uavs")[metric].mean().reset_index()
            plt.plot(agg["n_uavs"], agg[metric], marker="o", label=method)
        plt.xlabel("Number of UAVs")
        plt.ylabel(label)
        plt.title(f"{label} vs swarm size")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(Path(outdir) / f"scaling_{metric}.png", dpi=180)
        plt.close()

def plot_reliability_diagram(calib_df: pd.DataFrame, outpath: str) -> None:
    if calib_df is None or calib_df.empty:
        return
    plt.figure(figsize=(5.5, 5.5))
    plt.plot([0, 1], [0, 1], linestyle="--", label="ideal")
    plt.plot(calib_df["mean_score"], calib_df["empirical_rate"], marker="o", label="ACLD-v3")
    plt.xlabel("Mean warning score")
    plt.ylabel("Empirical event rate")
    plt.title("Reliability diagram")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=180)
    plt.close()

def build_manuscript_ready_tables(det_summary: pd.DataFrame,
                                  mit_summary: pd.DataFrame,
                                  sig_df: pd.DataFrame,
                                  outdir: str) -> None:
    Path(outdir).mkdir(parents=True, exist_ok=True)

    # Main detection table
    piv = det_summary.pivot_table(
        index=["scenario", "n_uavs", "density_scale_mode", "jnr_db"],
        columns="method",
        values=["pew_f1_mean", "fpr_mean", "edge_f1_mean", "runtime_median_ms_mean"]
    )
    piv.to_csv(Path(outdir) / "manuscript_main_detection_table.csv")

    # Closed-loop table
    mit_piv = mit_summary.pivot_table(
        index=["scenario", "n_uavs", "density_scale_mode", "jnr_db"],
        columns="controller",
        values=["partitions_mean", "connected_ratio_mean", "mean_lambda2_bin_mean", "actions_per_episode_mean"]
    )
    mit_piv.to_csv(Path(outdir) / "manuscript_closed_loop_table.csv")

    # Significance table
    sig_df.to_csv(Path(outdir) / "manuscript_significance_table.csv", index=False)


In [ ]:

# ============================================================
# Resume / checkpoint helpers (streaming writes, restart-safe)
# ============================================================

def _unlink_if_exists(path) -> None:
    path = Path(path)
    if path.exists():
        path.unlink()

def _load_csv_or_empty(path) -> pd.DataFrame:
    path = Path(path)
    if (not path.exists()) or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_csv(path)

def _append_rows_csv(path, rows: List[Dict]) -> None:
    if rows is None or len(rows) == 0:
        return
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(rows)
    write_header = (not path.exists()) or path.stat().st_size == 0
    df.to_csv(path, mode="a", header=write_header, index=False)

def _atomic_write_df(df: pd.DataFrame, path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def _write_resume_status(path, payload: Dict) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)

def _aggregate_detection_episode_df(ep_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate corrected detection episodes without collapsing runtime scopes.

    `runtime_incremental_*` is diagnostic method-wrapper latency after shared preprocessing
    and is explicitly NOT cross-method comparable. `acld_pipeline_*` is populated only
    for ACLD-v3 and matches the full `run_acld` scope used by C2. The legacy
    `runtime_median_ms*` aliases are retained only for backwards compatibility.
    """
    if ep_df.empty:
        return pd.DataFrame(columns=[
            "scenario", "method", "n_uavs", "density_scale_mode", "jnr_db",
            "n_episodes", "pew_f1_mean", "pew_f1_std", "fpr_mean", "fpr_std",
            "edge_f1_mean", "edge_f1_std", "edge_precision_micro_mean", "edge_recall_micro_mean",
            "edge_support_steps_mean", "cand_mean", "cand_std",
            "runtime_scope", "runtime_comparable_across_methods",
            "runtime_incremental_median_ms_mean", "runtime_incremental_median_ms_std",
            "runtime_incremental_p95_ms_mean", "runtime_incremental_p95_ms_std",
            "acld_pipeline_median_ms_mean", "acld_pipeline_median_ms_std",
            "acld_pipeline_p95_ms_mean", "acld_pipeline_p95_ms_std",
            "shared_context_median_ms_mean", "shared_context_median_ms_std",
            "shared_context_p95_ms_mean", "shared_context_p95_ms_std",
            "runtime_median_ms_mean", "runtime_median_ms_std",
            "top1_recall_mean", "top3_recall_mean", "top5_recall_mean",
            "prevalence_mean",
        ])
    return (ep_df.groupby(["scenario", "method", "n_uavs", "density_scale_mode", "jnr_db"], dropna=False)
              .agg(
                  n_episodes=("seed", "nunique"),
                  pew_f1_mean=("pew_f1", "mean"),
                  pew_f1_std=("pew_f1", "std"),
                  fpr_mean=("fpr", "mean"),
                  fpr_std=("fpr", "std"),
                  edge_f1_mean=("edge_f1_mean", "mean"),
                  edge_f1_std=("edge_f1_mean", "std"),
                  edge_precision_micro_mean=("edge_precision_micro", "mean"),
                  edge_recall_micro_mean=("edge_recall_micro", "mean"),
                  edge_support_steps_mean=("edge_support_steps", "mean"),
                  cand_mean=("candidate_edges_mean", "mean"),
                  cand_std=("candidate_edges_mean", "std"),
                  runtime_scope=("runtime_scope", "first"),
                  runtime_comparable_across_methods=("runtime_comparable_across_methods", "all"),
                  runtime_incremental_median_ms_mean=("runtime_incremental_median_ms", "mean"),
                  runtime_incremental_median_ms_std=("runtime_incremental_median_ms", "std"),
                  runtime_incremental_p95_ms_mean=("runtime_incremental_p95_ms", "mean"),
                  runtime_incremental_p95_ms_std=("runtime_incremental_p95_ms", "std"),
                  acld_pipeline_median_ms_mean=("acld_pipeline_median_ms", "mean"),
                  acld_pipeline_median_ms_std=("acld_pipeline_median_ms", "std"),
                  acld_pipeline_p95_ms_mean=("acld_pipeline_p95_ms", "mean"),
                  acld_pipeline_p95_ms_std=("acld_pipeline_p95_ms", "std"),
                  shared_context_median_ms_mean=("shared_context_median_ms", "mean"),
                  shared_context_median_ms_std=("shared_context_median_ms", "std"),
                  shared_context_p95_ms_mean=("shared_context_p95_ms", "mean"),
                  shared_context_p95_ms_std=("shared_context_p95_ms", "std"),
                  # Legacy alias retained for old plotting helpers only.
                  runtime_median_ms_mean=("runtime_median_ms", "mean"),
                  runtime_median_ms_std=("runtime_median_ms", "std"),
                  top1_recall_mean=("top1_recall_mean", "mean"),
                  top3_recall_mean=("top3_recall_mean", "mean"),
                  top5_recall_mean=("top5_recall_mean", "mean"),
                  prevalence_mean=("partition_next_rate", "mean"),
              )
              .reset_index())

def _aggregate_closed_loop_episode_df(ep_df: pd.DataFrame) -> pd.DataFrame:
    if ep_df.empty:
        return pd.DataFrame(columns=[
            "scenario", "controller", "intervention_mode", "n_uavs",
            "density_scale_mode", "jnr_db", "n_episodes", "partitions_mean",
            "partitions_std", "connected_ratio_mean", "connected_ratio_std",
            "mean_lambda2_bin_mean", "mean_lambda2_bin_std",
            "actions_per_episode_mean", "nodes_touched_total_mean",
            "dv_total_mean", "boost_db_total_mean",
        ])
    return (ep_df.groupby(["scenario", "controller", "intervention_mode", "n_uavs", "density_scale_mode", "jnr_db"], dropna=False)
              .agg(
                  n_episodes=("seed", "nunique"),
                  partitions_mean=("partitions", "mean"),
                  partitions_std=("partitions", "std"),
                  connected_ratio_mean=("connected_ratio", "mean"),
                  connected_ratio_std=("connected_ratio", "std"),
                  mean_lambda2_bin_mean=("mean_lambda2_bin", "mean"),
                  mean_lambda2_bin_std=("mean_lambda2_bin", "std"),
                  actions_per_episode_mean=("actions_per_episode", "mean"),
                  nodes_touched_total_mean=("nodes_touched_total", "mean"),
                  dv_total_mean=("dv_total", "mean"),
                  boost_db_total_mean=("boost_db_total", "mean"),
              )
              .reset_index())

def _completed_detection_seeds(ep_df: pd.DataFrame, spec: CampaignSpec) -> Set[int]:
    if ep_df.empty or "scenario" not in ep_df.columns:
        return set()
    sub = ep_df[ep_df["scenario"].astype(str) == str(spec.name)]
    if sub.empty:
        return set()
    need = len(tuple(spec.methods))
    counts = sub.groupby("seed")["method"].nunique().to_dict()
    return {int(seed) for seed, n_done in counts.items() if int(n_done) >= int(need)}

def _completed_closed_loop_keys(ep_df: pd.DataFrame, spec: CampaignSpec) -> Set[Tuple[str, int]]:
    if ep_df.empty or "scenario" not in ep_df.columns:
        return set()
    sub = ep_df[ep_df["scenario"].astype(str) == str(spec.name)]
    if sub.empty:
        return set()
    out = set()
    for _, row in sub[["controller", "seed"]].drop_duplicates().iterrows():
        out.add((str(row["controller"]), int(row["seed"])))
    return out

def run_detection_campaign(base_cfg: ACLDConfig,
                           specs: List[CampaignSpec],
                           outdir: str,
                           overwrite: bool = False,
                           store_step_rows: bool = True) -> Dict[str, pd.DataFrame]:
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    ep_path = outdir / "detection_episode_metrics_long.csv"
    st_path = outdir / "detection_step_metrics_long.csv"
    summary_path = outdir / "detection_summary_long.csv"
    status_path = outdir / "_resume_status_detection.json"

    if overwrite:
        for p in (ep_path, st_path, summary_path, status_path):
            _unlink_if_exists(p)

    ep_existing = _load_csv_or_empty(ep_path)

    total_units = int(sum(len(spec.seeds) for spec in specs))
    completed_units = 0

    for spec in specs:
        print(f"[A] scenario: {spec.name}")
        cfg = _clone_cfg(base_cfg, **spec.cfg_overrides)
        done_seeds = _completed_detection_seeds(ep_existing, spec)
        if done_seeds:
            print(f"    resume: {len(done_seeds)}/{len(spec.seeds)} seed already completed")

        for seed in spec.seeds:
            seed = int(seed)
            if seed in done_seeds:
                completed_units += 1
                print(f"    skip seed={seed} (already saved)")
                continue

            ep_rows, st_rows = run_detection_episode(
                cfg, spec.methods, spec.horizon_s, seed, store_step_rows=store_step_rows
            )

            episode_payload = []
            for r in ep_rows:
                rr = dict(r)
                rr["scenario"] = spec.name
                rr["n_uavs"] = int(cfg.n_uavs)
                rr["density_scale_mode"] = str(cfg.density_scale_mode)
                rr["jnr_db"] = cfg.jnr_db if cfg.jnr_db is not None else "NoJam"
                rr["alpha"] = float(cfg.alpha)
                rr["risk_band_fraction"] = float(cfg.risk_band_fraction)
                episode_payload.append(rr)

            step_payload = []
            for r in st_rows:
                rr = dict(r)
                rr["scenario"] = spec.name
                rr["seed"] = int(seed)
                rr["n_uavs"] = int(cfg.n_uavs)
                rr["density_scale_mode"] = str(cfg.density_scale_mode)
                rr["jnr_db"] = cfg.jnr_db if cfg.jnr_db is not None else "NoJam"
                rr["alpha"] = float(cfg.alpha)
                rr["risk_band_fraction"] = float(cfg.risk_band_fraction)
                step_payload.append(rr)

            _append_rows_csv(ep_path, episode_payload)
            if store_step_rows:
                _append_rows_csv(st_path, step_payload)

            ep_existing = pd.concat([ep_existing, pd.DataFrame(episode_payload)], ignore_index=True)
            done_seeds.add(seed)
            completed_units += 1
            _write_resume_status(status_path, {
                "campaign": "detection",
                "outdir": str(outdir.resolve()),
                "completed_seed_units": int(completed_units),
                "total_seed_units": int(total_units),
                "last_finished_scenario": str(spec.name),
                "last_finished_seed": int(seed),
                "store_step_rows": bool(store_step_rows),
            })
            print(f"    saved seed={seed} -> {len(done_seeds)}/{len(spec.seeds)} seeds done for scenario")

    ep_df = _load_csv_or_empty(ep_path)
    st_df = _load_csv_or_empty(st_path) if store_step_rows else pd.DataFrame()
    agg = _aggregate_detection_episode_df(ep_df)
    _atomic_write_df(agg, summary_path)
    return {"episode": ep_df, "step": st_df, "summary": agg}

def run_closed_loop_campaign(base_cfg: ACLDConfig,
                             specs: List[CampaignSpec],
                             controllers: Tuple[str, ...],
                             outdir: str,
                             intervention_mode: str = "hybrid",
                             overwrite: bool = False) -> Dict[str, pd.DataFrame]:
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    ep_path = outdir / "mitigation_episode_metrics_long.csv"
    st_path = outdir / "mitigation_step_metrics_long.csv"
    summary_path = outdir / "mitigation_summary_long.csv"
    status_path = outdir / "_resume_status_mitigation.json"

    if overwrite:
        for p in (ep_path, st_path, summary_path, status_path):
            _unlink_if_exists(p)

    ep_existing = _load_csv_or_empty(ep_path)

    total_units = int(sum(len(spec.seeds) * len(tuple(controllers)) for spec in specs))
    completed_units = 0

    for spec in specs:
        print(f"[B] mitigation scenario: {spec.name}")
        cfg = _clone_cfg(base_cfg, **spec.cfg_overrides)
        done_keys = _completed_closed_loop_keys(ep_existing, spec)
        if done_keys:
            print(f"    resume: {len(done_keys)}/{len(spec.seeds) * len(tuple(controllers))} controller-seed unit already completed")

        for controller in controllers:
            for seed in spec.seeds:
                seed = int(seed)
                key = (str(controller), seed)
                if key in done_keys:
                    completed_units += 1
                    print(f"    skip controller={controller}, seed={seed} (already saved)")
                    continue

                st_rows, summ = run_closed_loop_episode(
                    cfg, controller, spec.horizon_s, seed, intervention_mode=intervention_mode
                )

                step_payload = []
                for r in st_rows:
                    rr = dict(r)
                    rr["scenario"] = spec.name
                    rr["seed"] = int(seed)
                    rr["n_uavs"] = int(cfg.n_uavs)
                    rr["density_scale_mode"] = str(cfg.density_scale_mode)
                    rr["jnr_db"] = cfg.jnr_db if cfg.jnr_db is not None else "NoJam"
                    step_payload.append(rr)

                episode_payload = [dict(summ)]
                episode_payload[0]["scenario"] = spec.name
                episode_payload[0]["n_uavs"] = int(cfg.n_uavs)
                episode_payload[0]["density_scale_mode"] = str(cfg.density_scale_mode)
                episode_payload[0]["jnr_db"] = cfg.jnr_db if cfg.jnr_db is not None else "NoJam"

                _append_rows_csv(ep_path, episode_payload)
                _append_rows_csv(st_path, step_payload)

                ep_existing = pd.concat([ep_existing, pd.DataFrame(episode_payload)], ignore_index=True)
                done_keys.add(key)
                completed_units += 1
                _write_resume_status(status_path, {
                    "campaign": "mitigation",
                    "outdir": str(outdir.resolve()),
                    "completed_controller_seed_units": int(completed_units),
                    "total_controller_seed_units": int(total_units),
                    "last_finished_scenario": str(spec.name),
                    "last_finished_controller": str(controller),
                    "last_finished_seed": int(seed),
                    "intervention_mode": str(intervention_mode),
                })
                print(f"    saved controller={controller}, seed={seed}")

    ep_df = _load_csv_or_empty(ep_path)
    st_df = _load_csv_or_empty(st_path)
    agg = _aggregate_closed_loop_episode_df(ep_df)
    _atomic_write_df(agg, summary_path)
    return {"episode": ep_df, "step": st_df, "summary": agg}

def benchmark_runtime_vs_n(
    base_cfg: ACLDConfig,
    sizes: Iterable[int] = (50, 150, 500, 1000),
    repeats: int = 200,
    outdir: Optional[str] = None,
    overwrite: bool = False,
) -> List[Dict[str, float]]:
    if outdir is None:
        # Fallback: original in-memory behavior
        rows: List[Dict[str, float]] = []
        for n in sizes:
            cfg = ACLDConfig(**{**base_cfg.__dict__, "n_uavs": int(n)})
            rng = make_rng(cfg.seed + int(n))
            state = SwarmState.random_init(cfg, rng)
            acld = ACLDCore(cfg)

            times_ms = np.zeros(repeats, dtype=float)
            for k in range(repeats):
                t0 = time.perf_counter()
                _ = acld.run_acld(state)
                t1 = time.perf_counter()
                times_ms[k] = (t1 - t0) * 1000.0
                state.step_random_kinematics(cfg, rng)

            rows.append({
                "n": int(n),
                "pairs": int(n * (n - 1) // 2),
                "median_ms": float(np.median(times_ms)),
                "q1_ms": float(np.percentile(times_ms, 25)),
                "q3_ms": float(np.percentile(times_ms, 75)),
                "mean_ms": float(np.mean(times_ms)),
                "p95_ms": float(np.percentile(times_ms, 95)),
            })
        return rows

    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    progress_path = outdir / "runtime_scaling_progress.csv"
    final_path = outdir / "runtime_scaling_full.csv"
    status_path = outdir / "_resume_status_runtime.json"

    if overwrite:
        for p in (progress_path, final_path, status_path):
            _unlink_if_exists(p)

    total_units = int(len(tuple(sizes)) * int(repeats))
    completed_units = 0

    for n in sizes:
        n = int(n)
        progress_df = _load_csv_or_empty(progress_path)
        sub = progress_df[progress_df["n"] == n] if (not progress_df.empty and "n" in progress_df.columns) else pd.DataFrame()
        done_repeats = set(int(v) for v in sub["repeat_idx"].tolist()) if (not sub.empty and "repeat_idx" in sub.columns) else set()

        cfg = ACLDConfig(**{**base_cfg.__dict__, "n_uavs": n})
        rng = make_rng(cfg.seed + n)
        state = SwarmState.random_init(cfg, rng)
        acld = ACLDCore(cfg)

        done_prefix = 0
        while done_prefix in done_repeats:
            state.step_random_kinematics(cfg, rng)
            done_prefix += 1

        completed_units += len(done_repeats)
        if len(done_repeats):
            print(f"[C2] n={n}: resume from repeat {done_prefix}/{repeats}")

        for k in range(done_prefix, int(repeats)):
            t0 = time.perf_counter()
            _ = acld.run_acld(state)
            t1 = time.perf_counter()
            runtime_ms = (t1 - t0) * 1000.0

            _append_rows_csv(progress_path, [{
                "n": n,
                "repeat_idx": int(k),
                "runtime_ms": float(runtime_ms),
                "pairs": int(n * (n - 1) // 2),
            }])

            state.step_random_kinematics(cfg, rng)
            completed_units += 1
            _write_resume_status(status_path, {
                "campaign": "runtime_scaling",
                "outdir": str(outdir.resolve()),
                "completed_repeat_units": int(completed_units),
                "total_repeat_units": int(total_units),
                "last_finished_n": int(n),
                "last_finished_repeat_idx": int(k),
            })

    progress_df = _load_csv_or_empty(progress_path)
    if progress_df.empty:
        agg = pd.DataFrame(columns=["n", "pairs", "median_ms", "q1_ms", "q3_ms", "mean_ms", "p95_ms"])
    else:
        agg = (progress_df.groupby("n", as_index=False)
               .agg(
                   pairs=("pairs", "max"),
                   median_ms=("runtime_ms", "median"),
                   q1_ms=("runtime_ms", lambda s: float(np.percentile(np.asarray(s, dtype=float), 25))),
                   q3_ms=("runtime_ms", lambda s: float(np.percentile(np.asarray(s, dtype=float), 75))),
                   mean_ms=("runtime_ms", "mean"),
                   p95_ms=("runtime_ms", lambda s: float(np.percentile(np.asarray(s, dtype=float), 95))),
               )
               .sort_values("n")
               .reset_index(drop=True))
    _atomic_write_df(agg, final_path)
    return agg.to_dict(orient="records")


In [ ]:

# ============================================================
# Global experiment configuration
# ============================================================

OUTROOT = Path("acld_full_experiment_outputs_P0_corrected")
OUTROOT.mkdir(parents=True, exist_ok=True)

# Full paired seed sets
SEEDS_FULL = tuple(range(42, 62))      # 20 paired seeds
SEEDS_SCALE = tuple(range(142, 152))   # 10 paired seeds for scaling support
SEEDS_PILOT = (42, 43, 44)             # optional quick smoke subset

# Long horizons for manuscript-grade experiments
HORIZON_DET = 200.0
HORIZON_CL = 200.0

# Main method families
METHODS_FULL = tuple(MAIN_METHODS)
METHODS_ABLATION = tuple(ABLATION_METHODS)
CONTROLLERS_FULL = ("nh", "random", "tarjan_snapshot", "tarjan_predict", "acld")

BASE_CFG = ACLDConfig(
    n_uavs=150,
    include_ground_station=True,
    density_scale_mode="medium",
    jnr_db=None,
    alpha=0.0,
    beta=0.10,
    gamma=0.20,
    delta=0.10,
    zeta=0.05,
    risk_band_fraction=0.97,
    seed=42,
)

def make_spec(name, *, n_uavs, density, jnr=None, horizon_s=HORIZON_DET,
              seeds=SEEDS_FULL, methods=METHODS_FULL, rician_k_db=None,
              include_ground_station=None):
    overrides = dict(n_uavs=n_uavs, density_scale_mode=density, jnr_db=jnr)
    if include_ground_station is not None:
        overrides["include_ground_station"] = bool(include_ground_station)
    if rician_k_db is not None:
        m = rician_kdb_to_nakagami_m(float(rician_k_db))
        overrides["m_fading"] = m
        overrides["gs_m_fading"] = m
    return CampaignSpec(name=name, cfg_overrides=overrides, horizon_s=horizon_s,
                        seeds=tuple(seeds), methods=tuple(methods))

print("Output root:", OUTROOT.resolve())
print("Full seeds:", SEEDS_FULL[:3], "...", SEEDS_FULL[-3:])
print("Base config:", BASE_CFG)


# Resume/checkpoint control
OVERWRITE_EXISTING = False  # True -> delete prior outputs and rerun from scratch


### P0.4 reproducibility manifest

This cell captures the actual interpreter, package, operating-system, processor, source-revision, and protocol metadata from the environment that executes the corrected campaign. Manuscript environment claims must be taken from this generated manifest rather than typed manually.


In [ ]:
# ============================================================
# P0.4 environment / provenance manifest
# ============================================================
import sys
import platform
import subprocess
import importlib.metadata as _ilm

P0_PROTOCOL_VERSION = "P0.4-exact-bridge-microF1-runtime-scope"

def _pkg_version(name: str):
    try:
        return _ilm.version(name)
    except Exception:
        return None

def _git_head():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            stderr=subprocess.DEVNULL,
            text=True,
        ).strip()
    except Exception:
        return None

def capture_environment_manifest(outroot=OUTROOT):
    outroot = Path(outroot)
    outroot.mkdir(parents=True, exist_ok=True)
    manifest = {
        "p0_protocol_version": P0_PROTOCOL_VERSION,
        "python_version": sys.version,
        "python_executable": sys.executable,
        "platform": platform.platform(),
        "system": platform.system(),
        "release": platform.release(),
        "machine": platform.machine(),
        "processor": platform.processor(),
        "packages": {
            "numpy": _pkg_version("numpy"),
            "pandas": _pkg_version("pandas"),
            "scipy": _pkg_version("scipy"),
            "matplotlib": _pkg_version("matplotlib"),
        },
        "git_head": _git_head(),
        "source_revision_env": os.environ.get("ACLD_SOURCE_REVISION"),
        "output_root": str(outroot.resolve()),
        "runtime_scope_policy": {
            "cross_method_speed_claims": "forbidden unless identical timing boundaries are measured",
            "acld_full_pipeline": "ACLDCore.run_acld scope",
            "incremental_wrapper_latency": "diagnostic only; not cross-method comparable",
        },
        "edge_ground_truth_policy": {
            "definition": "exact Tarjan bridge set on realized next-step viable graph",
            "episode_metric": "micro-F1 from summed TP/FP/FN",
            "unsupported_steps": "tp=fp=fn=0 contributes zero counts and no artificial F1=0",
        },
    }
    path = outroot / "environment_manifest.json"
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)
    return manifest, path

P0_ENVIRONMENT_MANIFEST, P0_ENVIRONMENT_MANIFEST_PATH = capture_environment_manifest()
print(f"P0.4 environment manifest: {P0_ENVIRONMENT_MANIFEST_PATH}")
print(json.dumps(P0_ENVIRONMENT_MANIFEST, indent=2, ensure_ascii=False))


### P0 completion gate

Before manuscript tables are regenerated, every requested scenario/seed (or controller/seed) must be present. This gate prevents a partial campaign from being described as complete.


In [ ]:
def audit_detection_completion(ep_df: pd.DataFrame, specs: List[CampaignSpec]) -> pd.DataFrame:
    rows = []
    for spec in specs:
        sub = ep_df[ep_df["scenario"] == spec.name] if not ep_df.empty else pd.DataFrame()
        done = set(int(x) for x in sub["seed"].dropna().unique()) if (not sub.empty and "seed" in sub.columns) else set()
        expected = set(int(x) for x in spec.seeds)
        rows.append({
            "scenario": spec.name,
            "expected_seed_units": len(expected),
            "completed_seed_units": len(done & expected),
            "missing_seed_units": len(expected - done),
            "complete": len(expected - done) == 0,
        })
    return pd.DataFrame(rows)

def audit_mitigation_completion(ep_df: pd.DataFrame, specs: List[CampaignSpec], controllers: Tuple[str, ...]) -> pd.DataFrame:
    rows = []
    for spec in specs:
        sub = ep_df[ep_df["scenario"] == spec.name] if not ep_df.empty else pd.DataFrame()
        done = set((str(r.controller), int(r.seed)) for r in sub[["controller", "seed"]].drop_duplicates().itertuples()) if not sub.empty else set()
        expected = set((str(c), int(seed)) for c in controllers for seed in spec.seeds)
        rows.append({
            "scenario": spec.name,
            "expected_controller_seed_units": len(expected),
            "completed_controller_seed_units": len(done & expected),
            "missing_controller_seed_units": len(expected - done),
            "complete": len(expected - done) == 0,
        })
    return pd.DataFrame(rows)


## Experiment A1 — Main detection/localization campaign

Amaç:  
Ana iddiayı doğrudan test etmek: ACLD-v3, aynı senaryo ve aynı seed setinde baselines'a göre
- daha iyi **next-step partition early warning**
- daha iyi / rekabetçi **next-step edge localization**

sağlıyor mu?

Senaryolar:
- `n ∈ {150, 300, 500}`
- `density ∈ {sparse, medium, dense}`
- `JNR ∈ {NoJam, 5 dB, 15 dB}`
- channel = default Nakagami
- seeds = 20 paired seeds


In [ ]:

A1_OUT = OUTROOT / "A1_detection_core"
A1_specs = []
for n_uavs in (150, 300, 500):
    for density in ("sparse", "medium", "dense"):
        for jnr in (None, 5.0, 15.0):
            jam_tag = "NoJam" if jnr is None else f"J{int(jnr)}"
            A1_specs.append(
                make_spec(
                    f"A1_n{n_uavs}_{density}_{jam_tag}_Nakagami",
                    n_uavs=n_uavs,
                    density=density,
                    jnr=jnr,
                    horizon_s=HORIZON_DET,
                    seeds=SEEDS_FULL,
                    methods=METHODS_FULL,
                )
            )

A1_results = run_detection_campaign(BASE_CFG, A1_specs, outdir=A1_OUT, overwrite=OVERWRITE_EXISTING, store_step_rows=True)
plot_detection_summary(A1_results["summary"], A1_OUT / "figures")
A1_results["summary"].head(20)


## Experiment A2 — Rician/channel-degradation campaign

Amaç:  
Kanal bozulumu varken sonuçlar sürüyor mu?  
Bu hücre, “Nakagami altında iyi ama kanal stresi altında kırılıyor mu?” sorusunu kapatır.

Senaryolar:
- `n ∈ {150, 300, 500}`
- `density ∈ {sparse, medium, dense}`
- `Rician K ∈ {3 dB, 0 dB}`
- `JNR ∈ {NoJam, 15 dB}`
- seeds = 20 paired seeds


In [ ]:

A2_OUT = OUTROOT / "A2_detection_rician"
A2_specs = []
for n_uavs in (150, 300, 500):
    for density in ("sparse", "medium", "dense"):
        for k_db in (3.0, 0.0):
            for jnr in (None, 15.0):
                jam_tag = "NoJam" if jnr is None else f"J{int(jnr)}"
                A2_specs.append(
                    make_spec(
                        f"A2_n{n_uavs}_{density}_{jam_tag}_RicianK{int(k_db)}",
                        n_uavs=n_uavs,
                        density=density,
                        jnr=jnr,
                        horizon_s=HORIZON_DET,
                        seeds=SEEDS_FULL,
                        methods=METHODS_FULL,
                        rician_k_db=k_db,
                    )
                )

A2_results = run_detection_campaign(BASE_CFG, A2_specs, outdir=A2_OUT, overwrite=OVERWRITE_EXISTING, store_step_rows=True)
plot_detection_summary(A2_results["summary"], A2_OUT / "figures")
A2_results["summary"].head(20)


## Experiment A3 — ACLD ablation campaign

Amaç:  
ACLD-v3 içindeki hangi bileşen gerçekten gerekli?
- no guard
- no forecast bridge
- no adaptive threshold

Senaryolar:
- representative nominal medium
- representative dense nominal
- representative medium 15 dB
- seeds = 20 paired seeds


In [ ]:

A3_OUT = OUTROOT / "A3_ablation"
A3_specs = [
    make_spec("A3_medium_NoJam", n_uavs=150, density="medium", jnr=None,
              horizon_s=HORIZON_DET, seeds=SEEDS_FULL, methods=METHODS_ABLATION),
    make_spec("A3_dense_NoJam", n_uavs=150, density="dense", jnr=None,
              horizon_s=HORIZON_DET, seeds=SEEDS_FULL, methods=METHODS_ABLATION),
    make_spec("A3_medium_J15", n_uavs=150, density="medium", jnr=15.0,
              horizon_s=HORIZON_DET, seeds=SEEDS_FULL, methods=METHODS_ABLATION),
]

A3_results = run_detection_campaign(BASE_CFG, A3_specs, outdir=A3_OUT, overwrite=OVERWRITE_EXISTING, store_step_rows=True)
plot_detection_summary(A3_results["summary"], A3_OUT / "figures")
A3_results["summary"].head(20)


## Experiment B1 — Closed-loop mitigation (nominal)

Amaç:  
Makaledeki en kritik boşluğu kapatır:  
Alarm gerçekten müdahaleye bağlanırsa partition incident sayısı düşüyor mu?

Controller’lar:
- nh
- random
- tarjan_snapshot
- tarjan_predict
- acld

Senaryolar:
- `n ∈ {150, 300}`
- `density ∈ {sparse, medium, dense}`
- NoJam
- seeds = 20 paired seeds


In [ ]:

B1_OUT = OUTROOT / "B1_closed_loop_nominal"
B1_specs = []
for n_uavs in (150, 300):
    for density in ("sparse", "medium", "dense"):
        B1_specs.append(
            make_spec(
                f"B1_n{n_uavs}_{density}_NoJam",
                n_uavs=n_uavs,
                density=density,
                jnr=None,
                horizon_s=HORIZON_CL,
                seeds=SEEDS_FULL,
                methods=METHODS_FULL,
            )
        )

B1_results = run_closed_loop_campaign(
    BASE_CFG,
    B1_specs,
    controllers=CONTROLLERS_FULL,
    outdir=B1_OUT,
    intervention_mode="hybrid",
    overwrite=OVERWRITE_EXISTING,
)
plot_mitigation_summary(B1_results["summary"], B1_OUT / "figures")
B1_results["summary"].head(20)


## Experiment B2 — Closed-loop mitigation (harsh stress)

Amaç:  
Mitigation yalnız nominalde mi çalışıyor, yoksa ağır stres altında da işe yarıyor mu?

Senaryolar:
- `n ∈ {150, 300}`
- `density ∈ {sparse, medium, dense}`
- harsh stress = `JNR = 15 dB` + `Rician K = 0 dB`
- seeds = 20 paired seeds


In [ ]:

B2_OUT = OUTROOT / "B2_closed_loop_stress"
B2_specs = []
for n_uavs in (150, 300):
    for density in ("sparse", "medium", "dense"):
        B2_specs.append(
            make_spec(
                f"B2_n{n_uavs}_{density}_J15_RicianK0",
                n_uavs=n_uavs,
                density=density,
                jnr=15.0,
                horizon_s=HORIZON_CL,
                seeds=SEEDS_FULL,
                methods=METHODS_FULL,
                rician_k_db=0.0,
            )
        )

B2_results = run_closed_loop_campaign(
    BASE_CFG,
    B2_specs,
    controllers=CONTROLLERS_FULL,
    outdir=B2_OUT,
    intervention_mode="hybrid",
    overwrite=OVERWRITE_EXISTING,
)
plot_mitigation_summary(B2_results["summary"], B2_OUT / "figures")
B2_results["summary"].head(20)


## Experiment C1 — Accuracy scaling campaign

Amaç:  
Sadece runtime değil, **accuracy de swarm size ile birlikte** raporlansın.

Senaryolar:
- `n ∈ {50, 150, 300, 500, 1000}`
- `density = medium`
- `JNR ∈ {NoJam, 15 dB}`
- seeds = 10 paired seeds


In [ ]:

C1_OUT = OUTROOT / "C1_accuracy_scaling"
C1_specs = []
for n_uavs in (50, 150, 300, 500, 1000):
    for jnr in (None, 15.0):
        jam_tag = "NoJam" if jnr is None else f"J{int(jnr)}"
        C1_specs.append(
            make_spec(
                f"C1_n{n_uavs}_medium_{jam_tag}",
                n_uavs=n_uavs,
                density="medium",
                jnr=jnr,
                horizon_s=HORIZON_DET,
                seeds=SEEDS_SCALE,
                methods=METHODS_FULL,
            )
        )

C1_results = run_detection_campaign(BASE_CFG, C1_specs, outdir=C1_OUT, overwrite=OVERWRITE_EXISTING, store_step_rows=True)
plot_scaling_accuracy_runtime(C1_results["episode"], C1_OUT / "figures")
C1_results["summary"].head(20)


## Experiment C2 — Dedicated runtime scaling

Amaç:  
Runtime iddiasını ayrı ve doğrudan ölçmek.  
Bu hücre yalnız wall-clock benchmark üretir.


In [ ]:

C2_OUT = OUTROOT / "C2_runtime_scaling"
C2_OUT.mkdir(parents=True, exist_ok=True)

RUNTIME_SIZES = (50, 150, 300, 500, 1000)
RUNTIME_REPEATS = 1000

runtime_rows = benchmark_runtime_vs_n(BASE_CFG, sizes=RUNTIME_SIZES, repeats=RUNTIME_REPEATS, outdir=C2_OUT, overwrite=OVERWRITE_EXISTING)
runtime_df = pd.DataFrame(runtime_rows)
runtime_df.to_csv(C2_OUT / "runtime_scaling_full.csv", index=False)
_plot_runtime_scaling(runtime_rows, str(C2_OUT))
runtime_df


## Experiment C3 — GCS dependency sensitivity

Purpose: test whether detection/localization conclusions depend on including a fixed ground-station node. This directly separates the **network-model GCS node** from the **logical compute location** of ACLD.


In [ ]:
C3_OUT = OUTROOT / "C3_gcs_sensitivity"
C3_specs = []
for n_uavs in (150, 300):
    for jnr in (None, 15.0):
        jam_tag = "NoJam" if jnr is None else f"J{int(jnr)}"
        for include_gs in (True, False):
            gs_tag = "GCS_ON" if include_gs else "GCS_OFF"
            C3_specs.append(
                make_spec(
                    f"C3_n{n_uavs}_medium_{jam_tag}_{gs_tag}",
                    n_uavs=n_uavs,
                    density="medium",
                    jnr=jnr,
                    horizon_s=HORIZON_DET,
                    seeds=SEEDS_FULL,
                    methods=("acld_v3", "tarjan_predict_guard", "tarjan_snapshot_guard"),
                    include_ground_station=include_gs,
                )
            )

C3_results = run_detection_campaign(
    BASE_CFG, C3_specs, outdir=C3_OUT, overwrite=OVERWRITE_EXISTING, store_step_rows=True
)
plot_detection_summary(C3_results["summary"], C3_OUT / "figures")
C3_results["summary"]


## Experiment C4 — Parameterized telemetry payload overhead

This is an **assumption-based payload-only lower bound**, not measured radio traffic. It excludes MAC/PHY headers, retransmissions, acknowledgements, routing/control messages, security overhead, and synchronization traffic.


In [ ]:
def estimate_telemetry_payload_overhead(n_uavs: int, dt_ctrl: float,
                                        state_scalars_per_uav: int = 14,
                                        scalar_bytes: int = 4) -> Dict[str, float]:
    payload_per_uav = int(state_scalars_per_uav * scalar_bytes)
    payload_per_cycle = int(n_uavs * payload_per_uav)
    cycles_per_second = 1.0 / float(dt_ctrl)
    return {
        "n_uavs": int(n_uavs),
        "state_scalars_per_uav": int(state_scalars_per_uav),
        "scalar_bytes": int(scalar_bytes),
        "payload_bytes_per_uav_per_cycle": payload_per_uav,
        "payload_bytes_per_cycle": payload_per_cycle,
        "payload_kbit_per_second": payload_per_cycle * 8.0 * cycles_per_second / 1000.0,
        "scope": "payload-only lower-bound estimate; excludes protocol overhead",
    }

COMM_OVERHEAD_DF = pd.DataFrame([
    estimate_telemetry_payload_overhead(n, BASE_CFG.dt_ctrl)
    for n in (50, 150, 300, 500, 1000)
])
COMM_OVERHEAD_DF.to_csv(OUTROOT / "C4_payload_only_communication_overhead.csv", index=False)
COMM_OVERHEAD_DF


## Experiment D1 — Statistical significance

Amaç:  
“Daha iyi görünüyor” yerine gerçekten paired ve istatistiksel olarak daha iyi mi, onu göstermek.


In [ ]:

D1_OUT = OUTROOT / "D1_significance"
D1_OUT.mkdir(parents=True, exist_ok=True)

DET_EP_ALL = pd.concat(
    [A1_results["episode"], A2_results["episode"], C1_results["episode"]],
    axis=0,
    ignore_index=True,
)

SIG_DF = run_scenario_paired_significance_suite(
    DET_EP_ALL,
    ref_method="acld_v3",
    metrics=("pew_f1", "edge_f1_mean", "fpr"),
    outpath=D1_OUT / "scenario_balanced_paired_significance.csv",
)
SIG_DF.head(30)


## Experiment D2 — Prevalence-controlled jamming analysis

Amaç:  
Severe jamming’de performans artışı gerçekten robustness mı, yoksa görev kolaylaşması mı?


In [ ]:

D2_OUT = OUTROOT / "D2_prevalence_control"
D2_OUT.mkdir(parents=True, exist_ok=True)

STEP_FOR_PREV = pd.concat(
    [A1_results["step"], A2_results["step"]],
    axis=0,
    ignore_index=True,
)

PREV_DF = prevalence_control_analysis(
    STEP_FOR_PREV,
    method="acld_v3",
    group_col="scenario",
    n_repeats=200,
    seed=42,
)
PREV_DF.to_csv(D2_OUT / "balanced_prevalence_metrics.csv", index=False)
PREV_DF.groupby("scenario")[["balanced_f1", "balanced_precision", "balanced_recall", "balanced_fpr", "prevalence_original"]].mean()


## Experiment D3 — Calibration / reliability analysis

Amaç:  
ACLD warning score güvenilir mi? Reliability diagram ve ECE ile bakılır.


In [ ]:

D3_OUT = OUTROOT / "D3_calibration"
D3_OUT.mkdir(parents=True, exist_ok=True)

CAL_STEP = pd.concat(
    [A1_results["step"], A2_results["step"], C1_results["step"]],
    axis=0,
    ignore_index=True,
)

CAL_BINS, CAL_BY_SCENARIO = calibration_analysis(CAL_STEP, method="acld_v3", outdir=D3_OUT)
plot_reliability_diagram(CAL_BINS, D3_OUT / "reliability_diagram.png")

display(CAL_BINS)
display(CAL_BY_SCENARIO)


## Experiment D4 — Failure-mode analysis

Amaç:  
Hangi tip hatalar baskın?
- false negative
- false positive
- candidate explosion
- lambda2 over-warning


In [ ]:

D4_OUT = OUTROOT / "D4_failure_modes"
D4_OUT.mkdir(parents=True, exist_ok=True)

FAIL_STEP = pd.concat(
    [A1_results["step"], A2_results["step"], C1_results["step"]],
    axis=0,
    ignore_index=True,
)

FAIL_SUMMARY = failure_mode_analysis(
    FAIL_STEP,
    method="acld_v3",
    high_cand_quantile=0.95,
    outpath=D4_OUT / "failure_mode_summary.csv",
)
FAIL_SUMMARY.head(30)


## Experiment E — Build manuscript-ready outputs

Amaç:  
Makaledeki bütün tablolar ve figürler doğrudan bu campaign çıktılarından üretilecek.


In [ ]:

E_OUT = OUTROOT / "E_manuscript_ready"
E_OUT.mkdir(parents=True, exist_ok=True)

DET_SUMMARY_ALL = pd.concat(
    [A1_results["summary"], A2_results["summary"], A3_results["summary"], C1_results["summary"]],
    axis=0,
    ignore_index=True,
)

MIT_SUMMARY_ALL = pd.concat(
    [B1_results["summary"], B2_results["summary"]],
    axis=0,
    ignore_index=True,
)

build_manuscript_ready_tables(
    DET_SUMMARY_ALL,
    MIT_SUMMARY_ALL,
    SIG_DF,
    outdir=E_OUT,
)

DET_SUMMARY_ALL.to_csv(E_OUT / "all_detection_summaries.csv", index=False)
MIT_SUMMARY_ALL.to_csv(E_OUT / "all_mitigation_summaries.csv", index=False)
SIG_DF.to_csv(E_OUT / "all_significance.csv", index=False)
PREV_DF.to_csv(E_OUT / "all_prevalence_control.csv", index=False)
CAL_BY_SCENARIO.to_csv(E_OUT / "all_calibration_by_scenario.csv", index=False)
FAIL_SUMMARY.to_csv(E_OUT / "all_failure_modes.csv", index=False)

print("All manuscript-ready outputs written to:", E_OUT.resolve())
sorted(str(p.name) for p in E_OUT.iterdir())


## Notlar

- Bu notebook bilinçli olarak **tek bir küçük demo** değil, **tam kampanya** mantığıyla kuruldu.
- Eğer ilk turda yalnız çalışabilirlik testi yapmak isterseniz:
  - `SEEDS_FULL` yerine `SEEDS_PILOT`
  - `HORIZON_DET`, `HORIZON_CL` değerlerini küçültün.
- Makale metni yalnızca bu notebook’un ürettiği `E_manuscript_ready/` çıktılarından güncellenmelidir.
